# RSNA Knee Abnormality Detection — ensemble submission

Inference-only. Every model is loaded from an attached dataset; nothing is trained here.

**Pipeline** (four stages, each one rank-blended onto the previous):

| # | Stage | Models | Effect on `submission.csv` |
|---|-------|--------|----------------------------|
| 0 | benchmark | — | writes all-0.5 rows immediately, so a valid file always exists |
| 1 | DINOv2 pool | 20 × ViT-S/14 (+4 legacy folds if attached) | weighted rank mean → `submission.csv` |
| 2 | DINOv3 | 5 × ViT-S/16 folds | `0.55 × rank(stage 1) + 0.45 × rank(DINOv3)` |
| 3 | RadImageNet | frozen RN50 + 5 query heads | `0.65 × rank(stage 2) + 0.35 × rank(Rad)`, skipped for Baker's and Fracture |

**Credit.** Weights and label tables are other people's work, used as published:
`pilkwang` (20 DINOv2 members + LLM report labels), `mattiaangeli` (DINOv3 folds,
RadImageNet heads), `marwanmath` (RadImageNet ResNet-50 port), `stevenleehans`
and `lixin73` (independent report-label readings).

## Datasets to attach

Add all six through **+ Add Input** in the notebook editor. Names must match — the
code finds each one by walking `/kaggle/input`, so the mount path does not matter,
but the dataset contents do.

| Kind | Slug | Size | Provides |
|------|------|------|----------|
| Competition | `rsna-knee-abnormality-detection` | — | `test.csv`, `test_series/`, `train.csv` |
| Dataset | `pilkwang/rsna-knee-weights` | 1.7 GB | `manifest.json` + 20 × `m_*.pt` — **stage 1** |
| Dataset | `pilkwang/rsna-knee-llm-labels` | 0.1 MB | `report_labels_v2.csv` |
| Dataset | `mattiaangeli/knee-mri-fold-weights` | 439 MB | `m_f0..f4.pt` + timm wheel — **stage 2** |
| Dataset | `mattiaangeli/rsna-knee-radimagenet-foldsv1-heads` | 59 MB | `rad_head_f*.pt` + manifest — **stage 3** |
| Dataset | `marwanmath/resnet-50-radimagenet-marwan` | 94 MB | `ResNet50.pt` encoder — **stage 3** |
| Model | `metaresearch/dinov2` → PyTorch → `small` | 88 MB | DINOv2 backbone for stage 1 |

Notebook settings: **GPU T4 ×2**, **Internet OFF**, **Persistence: none**.

> `rsna_20260807_v1.pt` (4 extra "legacy" folds) is referenced but is not published
> anywhere. Its absence is handled — stage 1 runs with 20 members instead of 24.

In [ ]:
# ============================== CONFIGURATION ==============================
import time

WALL_T0 = time.time()

# Kaggle kills the kernel at 9 h. Each stage checks the clock before it starts;
# if the budget is gone it is skipped and the previous valid submission stands.
WALL_BUDGET_S = 9 * 3600
RESERVE_S     = 900        # keep 15 min spare for writing and committing

# MEASURED on T4 x2, running training studies as a pseudo test set (n = 3/120/160):
#   stage 1   54.4 s fixed + 4.930 s/study
#   stage 2   15.1 s fixed + 1.128 s/study
#   stage 3    3.3 s fixed + 0.385 s/study
#   combined  72.9 s fixed + 6.443 s/study
# -> 1300 studies ~= 2.35 h; 3000 ~= 5.39 h; the 9 h ceiling is not reached until
#    about 5,000 studies. Runtime is not the binding constraint on this pipeline.
# The cutoffs below are safety valves for a slower-than-expected mount, not a plan.
# Stage 1 has no internal deadline -- it is the one stage that could still run
# past 9 h and lose the commit, which is why it was calibrated most heavily.
DINOV3_START_CUTOFF_S = 6.0 * 3600
RAD_START_CUTOFF_S    = 7.5 * 3600

# Blend weights. Both are the values the source notebook was scored with.
A5_W_SETTING   = 0.45      # DINOv3 share in stage 2
RAD_ALPHA      = 0.35      # RadImageNet share in stage 3

# Stage 1 emits two candidates: a 20-member "public frontier" pool (no jitter TTA)
# and a jittered native pool that also includes the legacy folds when present.
# The source notebook promotes the public frontier as primary; True reproduces it.
# Flip to False to carry the native pool forward instead and A/B the two.
PROMOTE_PUBLIC_FRONTIER = True

# Report-label EDA: informative, costs ~40 s, and has no effect on predictions.
RUN_EDA = False

# Refuse to fall back to a 9-hour from-scratch training run if the weights
# dataset is missing. Leave True unless you deliberately want to train here.
REQUIRE_WEIGHTS = True


def wall_left():
    return WALL_BUDGET_S - RESERVE_S - (time.time() - WALL_T0)


def wall_log(msg):
    print(f'[wall {time.time() - WALL_T0:7.1f}s] {msg}', flush=True)

## Report lexicon

Multilingual rules that read the twelve findings out of a radiology report. Used only by the optional EDA below and by the from-scratch training fallback — the inference path does not touch it.

In [ ]:
from __future__ import annotations
import re
import unicodedata
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_PRE = str.maketrans({'ı': 'i', 'İ': 'i', 'I': 'i', 'ß': 'ss', 'đ': 'd', 'Đ': 'd', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae'})

def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join((ch for ch in text if not unicodedata.combining(ch)))
    text = text.replace('\xad', '')
    text = re.sub('[_\\-/\\\\]+', ' ', text)
    text = re.sub('[ \\t]+', ' ', text)
    return text
_SENT_SPLIT = re.compile('(?<=[.;!?])\\s+|\\n+')

def unwrap(text: str) -> str:
    if not isinstance(text, str):
        return ''
    out = []
    for line in text.split('\n'):
        s = line.strip()
        if out and out[-1] and (not re.search('[.;:!?>*•]$', out[-1])) and (len(out[-1].split()) >= 4) and s and (not s[:1].isupper()):
            out[-1] = out[-1] + ' ' + s
        else:
            out.append(s)
    return '\n'.join(out)

def clauses(text: str):
    norm = normalize(unwrap(text) if FEATURES['unwrap'] else text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(':') and len(c.split()) <= 14 and (i + 1 < len(raw)):
            merged.append(c + ' ' + raw[i + 1])
        merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend((p.strip() for p in c.split(',') if len(p.split()) > 2))
    return out
FEATURES = {'unwrap': True, 'directional_negation': True, 'oa_inherit': True, 'graded_pathology': True, 'synovitis_backoff': True}

def _rx(*alts: str) -> re.Pattern:
    return re.compile('|'.join(alts))
PRE_NEG = _rx('\\bno\\b', '\\bnot\\b', '\\bwithout\\b', '\\bnegative for\\b', '\\babsence\\b', '\\bno evidence\\b', '\\bfree of\\b', '\\bnone\\b', '\\bneither\\b', '\\bnor\\b', '\\bsin\\b', '\\bno hay\\b', '\\bausencia\\b', '\\bausentes?\\b', '\\bno se\\b', '\\bpas de\\b', '\\bsans\\b', '\\baucune?\\b', '\\bgeen\\b', '\\bzonder\\b', '\\bniet\\b', '\\bkeine?[nmrs]?\\b', '\\bohne\\b', '\\bnicht\\b', '\\bkein\\b', '\\bnema\\b', '\\bbez\\b', '\\bnisu\\b', '\\bnije\\b', '\\bδεν\\b', '\\bχωρις\\b', 'ουδεν', '\\bουτε\\b', '\\bбез\\b', '\\bне\\b', 'липсва', '\\bняма\\b')
POST_NEG = _rx('\\byok\\b', '\\byoktur\\b', 'izlenmemekte', 'saptanmadi', '\\bdegil\\b', 'gozlenmemekte', 'mevcut degil', 'eslik etmiyor', '\\bizlenmedi\\b', 'izlenmemistir', 'saptanmamistir', 'gorulmemistir', '\\bnema znakova\\b', 'bez znakova')
NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, '\\bunremarkable\\b')
NEG_WINDOW = 90

def _negated(clause: str, start: int, end: int) -> bool:
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            if not re.search('\\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|ωστοσο|αλλα|но)\\b', clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    return False
NORMALITY = _rx('\\bnormal', '\\bintact\\b', '\\bpreserved\\b', '\\bwithin normal limits\\b', 'limites normales', '\\bconservad', '\\bintegr', '\\bnormales\\b', '\\bdoga(l|ll)\\b', 'korunmus', '\\bnormaldir\\b', 'olagan', '\\buredn', '\\bocuvan', '\\bodrzan', '\\bintakt', '\\bprimjeren', '\\bodrzanog kontinuiteta', '\\bodržan', 'φυσιολογικ', 'ακεραι', 'δεν παρατηρουνται', 'δεν σημειωνονται', 'unauffallig', 'regelrecht', '\\bo\\.?b\\.?\\b', 'нормал', 'запазен', 'съхранен', '\\bбез особености\\b', 'интактн', '\\bgaaf\\b', '\\bnormaal\\b')
NORMAL_PHRASE = _rx('\\bsin alteracion', '\\bsin cambios\\b', '\\bsin particularidad', '\\bsin hallazgos\\b', '\\bsin lesion', '\\bsin signos de (rotura|lesion)', '\\bcontinu[oa]s?\\b', '\\bcontinuidad conservada\\b', '\\bno abnormalit', '\\bno significant abnormalit', '\\bunremarkable\\b', '\\bno evidence of (tear|injury|abnormalit)', '\\bohne auffalligkeit', '\\bkein nachweis\\b', '\\bohne befund\\b', '\\bgeen afwijking', '\\bzonder afwijking', '\\bsans anomalie', "\\bpas d[e']anomalie", '\\bbez osobitosti\\b', '\\bbez znakova (rupture|lezije)\\b', '\\bbez patoloskih\\b', 'χωρις αλλοιωσ', 'χωρις παθολογ', 'δεν παρατηρουνται (αξιολογα|παθολογ)', '\\bбез особености\\b', '\\bбез патологич', '\\bбез данни за\\b', '\\bozel bir ozellik yok', '\\bpatolojik bulgu (yok|izlenmemis)')
UNCERTAIN = _rx('\\bpossible\\b', '\\bprobable\\b', '\\bsuspicious\\b', '\\bsuspected?\\b', 'cannot (be )?exclude', '\\bmay\\b', '\\bquestionable\\b', '\\bequivocal\\b', '\\br/o\\b', '\\bdd\\b', '\\blikely\\b', '\\bsuggest', '\\bcompatible with\\b', '\\bposible\\b', 'sin criterios categoricos', '\\bdudos', '\\bsugier', '\\bmuhtemel\\b', '\\bolasi\\b', '\\bsupheli\\b', '\\bizlenim', '\\bdusundur', '\\bmoguce\\b', '\\bvjerojatno\\b', '\\bsumnja\\b', '\\bmoze odgovarati\\b', 'πιθαν', 'υποπτ', '\\bmoglich', '\\bverdachtig', '\\bfraglich', '\\bv\\.?a\\.?\\b', '\\bwohl\\b', '\\bвъзможно\\b', '\\bвероятно\\b', 'суспект', '\\bmogelijk\\b', '\\bverdacht\\b')

In [ ]:
TEAR = _rx('\\btear', '\\btorn\\b', '\\brupture', '\\bdisruption\\b', 'discontinuit', '\\bavuls', '\\bmacerat', '\\bbuckethandle\\b', 'bucket handle', '\\brotura\\b', '\\broturas\\b', '\\bruptura', '\\bdesgarro', '\\broto\\b', '\\bdechirure', '\\bdechire', '\\bscheur', '\\bruptuur', 'gescheurd', '\\briss\\b', 'einriss', '\\bruptur', 'zerreiss', '\\blasion', '\\bausriss', '\\byirtik', '\\byirtig', '\\bkopma\\b', 'butunluk kaybi', '\\brupturu\\b', 'devamsizlik', '\\brupture\\b', '\\bdevamliligi secilememis', '\\bpuknuce', '\\bprekid\\b', '\\bpukotin', '\\bruptur', 'ρηξη', 'ρηξις', 'ρηγμα', 'ασυνεχεια', 'руптура', 'разкъсв', 'разрив', 'скъсв', '\\bлезия\\b')
DEGEN = _rx('degenerat', '\\bmucoid\\b', '\\bmyxoid\\b', '\\bfray', '\\bfissur', 'dejeneratif', '\\bmukoid\\b', 'degenerativn', 'εκφυλ', 'дегенерат', '\\bμυξοειδ', '\\bμυξωδ', '\\bmeniskopat', '\\bmeniscopath', '\\bmuco ?ide\\b', 'aufgefasert', '\\bdejenerasyon\\b')
INJURY = _rx('\\binjur', '\\bsprain', '\\blesion', '\\blasion', '\\bedema\\b', '\\boedema\\b', '\\bodem\\b', '\\bedem\\b', '\\bοιδημα', '\\bодем', '\\bедем', '\\bstrain\\b', '\\bhigh signal\\b', '\\bsignal alteration\\b', '\\bhiperintens', '\\bhyperintens', 'aumento de senal', 'alteracion de senal', 'cambio de senal', '\\bsignalanhebung', '\\bsignalalteration', 'verhoogd signaal', 'sinyal artis', 'αυξημενο σημα', 'повишен сигнал', '\\besguince\\b', '\\bthicken', '\\bzadebljanje\\b', '\\bverdikking\\b', '\\bdistenzij', '\\blaksite\\b', '\\blaxity\\b', '\\bpartial\\b', '\\bparcijaln', '\\bparcial', '\\bpartiel', '\\bpartiell')
_GRADE_RX = re.compile('(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)[\\s:]*(?:grade\\s*)?([1-4]|iv|iii|ii|i)\\b')
_ROMAN = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4}

def _grade_of(clause: str):
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best
ANAT = {'ACL': _rx('anterior cruciate', '\\bacl\\b', 'cruzado anterior', '\\blca\\b', 'croise anterieur', 'voorste kruisband', '\\bvkb\\b', 'vorderes kreuzband', 'vorderen kreuzband', 'vordere kreuzband', 'on capraz', '\\bocb\\b', 'anterior capraz', 'prednji krizni', 'prednjeg krizn', 'προσθι[οα][^ ]* χιαστ', 'προσθιου χιαστου', 'χιαστο[^ ]* συνδεσμ', '\\bχιαστ\\w*', 'предна кръстна', 'предната кръстна', 'предна кръста', 'cruciate ligaments', 'ligamentos cruzados', 'ligaments croises', 'kruisbanden', 'kreuzbander', 'capraz baglar', 'krizn[a-z]* ligament[a-z]*', 'χιαστοι συνδεσμ', 'χιαστων συνδεσμ', 'кръстните връзки', 'кръстни връзки'), 'MCL': _rx('medial collateral', '\\bmcl\\b', 'tibial collateral', 'colateral medial', 'colateral interno', '\\blcm\\b', 'collateral medial', 'collateral interne', 'mediale collaterale', 'binnenband', '\\b(mediale|laterale) banden\\b', '\\bcollaterale banden\\b', 'innenband', 'mediales? kollateral', '\\bic yan bag', 'medial kollateral', '\\biyb\\b', 'medyal kollateral', 'medijalni kolateraln', 'medijalnog kolateraln', 'εσω πλαγι', 'εσωτερικο πλαγι', '\\bπλαγι\\w* συνδεσμ', '\\bπλαγιοι\\b', 'медиален колатерал', 'вътрешна странична', '\\bколатерал\\w*', '\\bcolaterales\\b', '\\bcollateraux\\b', '\\bcollateralen\\b', '\\bkolateralni\\b', 'collateral ligaments', 'ligamentos colaterales', 'ligaments collateraux', 'collaterale banden', 'kollateralbander', 'seitenbander', 'yan baglar', 'kolateraln[a-z]* ligament[a-z]*', 'πλαγιοι συνδεσμ', 'πλαγιων συνδεσμ', 'колатерални връзки', 'страничните връзки'), 'Medial Meniscus': _rx('medial meniscus', '\\bmm\\b(?= tear)', 'medial menisc', 'menisco medial', 'menisco interno', 'menisque medial', 'menisque interne', 'mediale meniscus', 'binnenmeniscus', 'innenmeniskus', 'medialen? meniskus', 'innenmeniskushinterhorn', 'medyal menisk', '\\bic menisk', 'medijalni meniskus', 'medijalnog meniskusa', 'medijalnom meniskusu', 'medijaln\\w* menisk\\w*', '\\bmedijalnog meniska\\b', 'medijalni menisk', 'εσω μηνισκ', 'μηνισκ[^ ]* του εσω', 'εσω διαμερισμα[^.]{0,40}μηνισκ', 'медиалния менискус', 'медиален менискус', 'вътрешния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'amfoteroi\\w* mhnisk', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc'), 'Lateral Meniscus': _rx('lateral meniscus', 'lateral menisc', 'menisco lateral', 'menisco externo', 'menisque lateral', 'menisque externe', 'laterale meniscus', 'buitenmeniscus', 'aussenmeniskus', 'lateralen? meniskus', 'aussenmeniskushinterhorn', 'lateral menisk', '\\bdis menisk', 'lateralni meniskus', 'lateralnog meniskusa', 'lateralnom meniskusu', 'lateraln\\w* menisk\\w*', '\\blateralnog meniska\\b', 'εξω μηνισκ', 'μηνισκ[^ ]* του εξω', 'εξω διαμερισμα[^.]{0,40}μηνισκ', 'латералния менискус', 'латерален менискус', 'външния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc')}
OA_EVIDENCE = _rx('osteoarthrit', '\\barthros', '\\bgonarthros', '\\bosteoarthros', 'chondropath', 'chondromalac', 'condropat', 'condromalac', '\\bchondros', '\\bchondrosis\\b', 'chondral (loss|defect|ulcer|thinning|injury|fissur|wear)', 'cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)', '(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage', 'articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)', 'osteophyt', 'osteofit', 'osteofyt', 'osteofito', 'osteophyten', 'spurring', 'joint space narrowing', 'pinzamiento articular', 'reduced joint space', 'kikirdak kayb', 'kikirdak incelme', 'kondropati', 'kondral', 'kikirdak dejener', 'eklem aralig\\w* daral', 'eklem mesafesi daral', 'kikirdak kalinlig\\w* azal', 'kraakbeen', 'gonartrose', 'artrose', '\\bknorpel', 'arthrose', 'gonarthrose', 'hrskavic', 'hondromalac', 'artroz', 'osteoartrit', 'artrotsk', 'artrotick', '\\boa promjen', '\\boa\\b', 'degenerativne promjene hrskav', 'χονδρ[^ ]*παθ', 'αρθριτ', 'αρθρωσ', 'οστεοφυτ', 'χονδρομαλακ', 'αρθρικου χονδρου', 'εξαλειψη του αρθρικου χονδρου', 'διαβρωση του αρθρικου χονδρ', 'λεπτυνση[^.]{0,30}χονδρ', 'φθορα[^.]{0,20}χονδρ', 'артроз', 'хондропат', 'остеофит', 'хрущял[^.]{0,40}(изтън|увред|дефект|липс)', 'изтъняване[^.]{0,30}хрущял', 'хондромалац', 'ulcera[s]? condral', 'cartilago[^.]{0,25}(perdida|adelgaz)', 'icrs grade', 'icrs\\b', 'outerbridge', '\\bdenudation\\b', 'denudacij', 'erozivne promjene', '\\berosion of[^.]{0,20}cartilage', 'kraakbeenlijden', 'kraakbeenverlies')
TF_SITE = _rx('compartment', 'compartimento', 'compartiment', 'kompartman', 'kompartiment', 'kompartment', 'odjelj', 'διαμερισμα', 'компартм', '\\bотдел', 'femorotibial', 'tibiofemoral', 'femoro tibial', 'femorotibiaal', 'femorotibijaln', 'феморотибиал', '\\bft zglob', 'tibiofemoraln', 'condyle', 'condilo', 'kondyl', 'kondil', 'condyl', 'κονδυλ', 'кондил', '\\bplateau', '\\bplato\\b', 'platillo', 'meseta', 'плато', 'tibiaplateau', 'tibijaln\\w* plato', 'tibyal plato', 'tibia plato', 'κνημιαι', 'μηριαι', 'weightbearing', 'weightbaring', 'zona de carga', 'dragende deel', 'agirlik tasiyan', '\\bfemur\\b', '\\btibia\\b', '\\bfemoral\\b', '\\btibial\\b', '\\bfemura\\b', '\\btibije\\b', '\\bmesarthrio\\b', 'μεσαρθριο')
PF_SITE = _rx('patellofemoral', 'femoropatellar', 'femoropatelar', 'patelofemoral', 'retropatellar', 'retrorotulian', 'trochlea', 'troclea', 'troklea', 'trochlear', 'trohlej', 'τροχιλ', '\\bpatella', '\\bpatellar', 'rotulian', '\\brotula\\b', '\\bpatele\\b', 'patellofemoraal', 'femoropatellair', 'επιγονατιδ', 'μηροεπιγονατιδ', 'пател', 'феморопател', 'anterior compartment', 'compartimento anterior', 'prednj\\w* odjeljk', '\\bfp zglob', '\\bpf zglob', '\\bfaset', '\\bfacet', 'patellofemoraln')
SIDE_MEDIAL = _rx('\\bmedial\\w*', '\\bmedyal\\w*', '\\bmedijaln\\w*', '\\bmediaal\\w*', '\\bmediale\\w*', '\\binterno\\b', '\\binterna\\b', '\\binternos\\b', '\\binterne\\b', '\\binnen\\w*', '\\bic\\b', '\\bunutarnj\\w*', '\\bεσω\\w*', '\\bεσωτερικ\\w*', '\\bмедиал\\w*', '\\bвътреш\\w*', '\\bbinnen\\w*', '\\bmediaal\\b', '\\bmediales?\\b')
SIDE_LATERAL = _rx('\\blateral\\w*', '\\bexterno\\b', '\\bexterna\\b', '\\bexternos\\b', '\\bexterne\\b', '\\bdis\\b', '\\blateraln\\w*', '\\baussen\\w*', '\\bbuiten\\w*', '\\bεξω\\w*', '\\bεξωτερικ\\w*', '\\bлатерал\\w*', '\\bвъншн\\w*', '\\bvanjsk\\w*')
SIDE_ANTERIOR = _rx('\\banterior\\w*', '\\bant\\b', '\\bon\\b', '\\bprednj\\w*', '\\bvorder\\w*', '\\bvoorste\\b', '\\bπροσθι\\w*', '\\bпредн\\w*', '\\banteriyor\\w*', '\\bavant\\b', '\\banterieur\\w*')
GLOBAL_OA = _rx('tri ?compartment', 'all three compartment', 'global(ised)? (oa|osteoarthrit)', '\\bgonarthros', '\\bgonartros', '\\bgonarthrose', '\\bgonartrose', 'gonartro', 'goanrtrot', 'gonartrot', 'osteoarthritis of the knee', 'artrosis (de |)(la )?rodilla', 'knee osteoarthrit', '\\bdiz osteoartrit', '\\bgonartroz', 'artroza koljena', 'οστεοαρθριτιδα', 'αρθριτιδα του γονατος', 'εκφυλιστικη οστεοαρθριτ', 'артроза на колянната', 'гонартроз', 'degenerative joint disease', '\\bdjd\\b', 'three compartments', 'compartmens', 'compartments')
DIRECT = {'Effusion': _rx('\\beffusion', 'joint fluid', 'intra ?articular fluid', '\\bhydrops\\b', '\\bhemarthros', '\\bhaemarthros', 'derrame articular', '\\bderrame\\b', 'liquido articular', 'hemartrosis', 'epanchement', 'gewrichtsvocht', '\\bvocht\\b', 'gewrichtseffusie', 'opzetting van suprapatell', 'gelenkerguss', '\\berguss\\b', 'gelenksergu', 'gelenksflussigkeit', 'eklem\\w* ic\\w* sivi', 'efuzyon', 'eklem sivisi', 'eklem mesafesinde sivi', 'sivi (miktari|artisi|birikimi)', 'sivi artis', '\\bsivi\\b[^.]{0,25}artmis', '\\bizljev', '\\bizliv', 'zglobn[^ ]* tekucin', '\\bhidrops\\b', 'αρθρικ[^ ]* υγρ', 'υγρου ενδαρθρικα', 'ενδαρθρικ[^ ]* υγρ', 'ποσοτητα υγρου', 'ενδαρθρικ', 'αρθρικη συλλογη', 'υγρο στην αρθρωση', 'υγρου στην αρθρωση', 'συλλογη υγρου', 'ενθαρθρικ', 'ставен излив', 'излив', 'ставна течност', 'синовиална течност'), 'Synovitis': _rx('synovit', 'sinovit', 'synovial (thickening|proliferation|hypertroph)', 'thicken\\w* synovial', 'hypertroph\\w* of the synovium', 'synoviale? (verdikking|proliferatie)', 'verdikkingen van (het )?synovium', 'synovialitis', 'synovialis(verdickung|proliferation)', 'reizsynovial', 'sinovijalitis', 'sinovitis', 'zadebljanje sinovij', 'proliferacij\\w* sinovij', 'sinovijaln\\w* proliferacij', 'υμενιτιδα', 'συνοβιτιδα', 'υμενικ[^ ]* υπερτροφ', 'αρθρικου υμεν', 'παχυνση[^.]{0,20}υμεν', 'υμενα', 'синовит', 'синовиал[^ ]* (задебел|пролифер)', '\\bpannus\\b', '\\bhoffit', 'sinovyal\\w* (kalinlas|proliferas)', 'sinovyal hipertrof', '\\bartrit\\b', '\\barthritis\\b'), "Baker's": _rx('baker', 'popliteal cyst', 'quiste popliteo', 'quistes popliteos', 'kyste poplite', 'popliteale? cyst', 'poplitealzyste', 'bakerzyste', 'popliteal kist', '\\bbakerova\\b', 'poplitealn[^ ]* cist', 'popliteal\\w* cist', 'κυστη baker', 'πολυχωρη συνοβιακη κυστη', 'κυστη του baker', 'συνοβιακη κυστη', 'κυστη τυπου baker', 'киста на бейкър', 'бейкърова киста', 'поплитеална киста', 'бекеров', 'gastrocnemio ?semimembranos', 'gastrocnemius semimembranosus burs'), 'Contusion': _rx('\\bcontusion', 'bone bruise', 'bone marrow (o?edema|contusion)', 'marrow o?edema', '\\bkontuz', 'medular bone o?edema', 'osseous contusion', 'contusion osea', 'edema oseo', 'edema de medula osea', 'contusiones oseas', 'oedeme osseux', 'contusion osseuse', 'botcontusie', 'botoedeem', 'beenmergoedeem', 'botmergoedeem', 'knochenmarkodem', 'knochenodem', 'knochenmarksodem', 'kontusion', 'kemik kontuzyonu', 'kemik iligi odemi', 'kemik odemi', 'kemik iliginde odem', 'kontuzyonel kemik', 'kemik iligi odemleri', 'kostani edem', 'edem kosti', 'kontuzij', 'kostane srzi[^.]{0,20}edem', 'οστεομυελικ[^ ]* οιδημα', 'οστικο οιδημα', 'μυελικο οιδημα', 'οστικο μωλωπ', 'костномозъчен едем', 'костен едем', 'контузионен', 'костно мозъчен едем'), 'Fracture': _rx('\\bfractur', '\\bfract\\b', '\\bfractura', '\\bfracturas\\b', '\\bfractuur', '\\bbreuk\\b', '\\bfraktur', '\\bbruch\\b', '\\bkirik\\b', '\\bkirigi\\b', '\\bkiri[kg]\\w*', '\\bprijelom', 'impresijsk[^ ]* fraktur', 'impaktcij', 'καταγμα', 'καταγματ', 'фрактур', 'счупван', 'фисур', 'insufficiency fracture', 'stress fracture', 'avulsion fracture', 'subchondral fracture', 'subkondral kiri', 'impaction (fracture|injury)', 'osteochondral (fracture|impaction)', '\\bsegond\\b', 'impactiefractuur', 'subchondrale impression', 'subchondraler? impress')}
DECOY = {'Fracture': _rx('microfractur', '\\bfracture (risk|prophyla)'), "Baker's": _rx('meniscal cyst', 'quiste meniscal', 'parameniscal')}
PAIRED = {'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus'}
OA_TARGETS = ['Medial OA', 'Lateral OA', 'PF OA']
PLURAL_MENISCI = _rx('\\bmenisci\\b', '\\bmeniscos\\b', '\\bmenisques\\b', '\\bmenisken\\b', '\\bmeniskusi\\b', '\\bmenisk\\w*ler\\b', '\\bμηνισκοι\\b', '\\bμηνισκων\\b', '\\bменискуси\\b', '\\bменискусите\\b', '\\bmenisci\\w*\\b')
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)
STEM_MENISCUS = _rx('menisc\\w*', 'menisk\\w*', 'μηνισκ\\w*', 'мениск\\w*')
STEM_CRUCIATE = _rx('cruciate', 'cruzado', 'croise', 'kruisband', 'kreuzband', 'capraz bag\\w*', 'krizn\\w*', 'χιαστ\\w*', 'кръстн\\w*', '\\bacl\\b', '\\blca\\b', '\\bvkb\\b', '\\bocb\\b', '\\bacb\\b')
STEM_COLLATERAL = _rx('collateral\\w*', 'colateral\\w*', 'kollateral\\w*', 'collaterale\\w*', 'kolateraln\\w*', 'yan bag\\w*', 'πλαγι\\w*', 'колатерал\\w*', 'странич\\w*', 'innenband\\w*', 'binnenband\\w*', '\\bmcl\\b', '\\blcm\\b', '\\biyb\\b')
STEM_FRACTURE = _rx('fractur\\w*', 'fraktur\\w*', 'fractuur\\w*', '\\bfract\\b', 'kiri[kgğ]\\w*', 'prijelom\\w*', 'lom kosti', '\\bbreuk\\w*', '\\bbruch\\w*', 'καταγμα\\w*', 'καταγματ\\w*', 'фрактур\\w*', 'счупван\\w*', 'fisur\\w* (osea|oseas|kost)', 'fissur\\w* kost')
POSTERIOR_ONLY = _rx('\\bpcl\\b', '\\blcp\\b', '\\bhkb\\b', '\\bacb\\b', 'posterior cruciate', 'cruzado posterior', 'croise posterieur', 'achterste kruisband', 'hinteres kreuzband', 'arka capraz', 'straznji krizn', 'οπισθι[οα]\\w* χιαστ', 'задна кръстн', 'задната кръстн')
LATERAL_COLL_ONLY = _rx('\\blcl\\b', '\\bfcl\\b', 'lateral collateral', 'fibular collateral', 'colateral lateral', 'colateral externo', 'buitenband', 'aussenband', 'dis yan bag', 'lateralni kolateraln', 'εξω πλαγι', 'латерален колатерал')

In [ ]:
def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int=55):
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False
STEM_RULES = {'ACL': (STEM_CRUCIATE, SIDE_ANTERIOR), 'MCL': (STEM_COLLATERAL, SIDE_MEDIAL), 'Medial Meniscus': (STEM_MENISCUS, SIDE_MEDIAL), 'Lateral Meniscus': (STEM_MENISCUS, SIDE_LATERAL)}

class _Matcher:

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None
ANAT_MATCH = {t: _Matcher(ANAT[t], *STEM_RULES[t]) for t in PAIRED}
DIRECT_MATCH = {t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == 'Fracture' else rx) for t, rx in DIRECT.items()}
SEV_LOW = _rx('\\bsmall\\b', '\\bminimal\\b', '\\btrace\\b', '\\bmild\\b', '\\bslight\\b', '\\btiny\\b', '\\bscant\\b', '\\bdiscrete\\b', '\\blow ?grade\\b', '\\bincipient\\b', '\\bleve\\b', '\\bminim', '\\bpeque', '\\bfina\\b', '\\bfino\\b', '\\bligero\\b', '\\bescaso\\b', '\\bdiscreto\\b', '\\bhafif\\b', '\\baz miktarda\\b', '\\bsilik\\b', '\\bmanj\\w*', '\\bblago\\b', '\\bdiskretn', '\\bmalo\\b', '\\bpocetn', '\\bgering', '\\bdiskret', '\\bkleine?r?\\b', '\\bwenig\\b', '\\bzarte?\\b', '\\bbeperkte?\\b', '\\bgeringe\\b', '\\bweinig\\b', '\\blichte?\\b', '\\blicht\\b', '\\bηπι', '\\bμικρ', '\\bελαχιστ', '\\bαρχομεν', '\\bминимал', '\\bлек', '\\bмалк', '\\bнеголям')
SEV_HIGH = _rx('\\blarge\\b', '\\bmarked\\b', '\\bmassive\\b', '\\bsevere\\b', '\\bextensive\\b', '\\bmoderate\\b', '\\bgross\\b', '\\bsignificant\\b', '\\babundant\\b', '\\btense\\b', '\\bcomplete\\b', '\\bfull ?thickness\\b', '\\bhigh ?grade\\b', '\\badvanced\\b', '\\bmoderad', '\\bimportante\\b', '\\bsevera?\\b', '\\bmarcad', '\\bcuantios', '\\bespesor total\\b', '\\bcompleta?\\b', '\\bbelirgin\\b', '\\byaygin\\b', '\\bileri\\b', '\\bciddi\\b', '\\bbol\\b', '\\bkomplet', '\\bopsezan\\b', '\\bveliki\\b', '\\bizrazit', '\\bznacajn', '\\bumjeren', '\\buznapredoval', '\\bpotpun', '\\bkompleksn', '\\bausgepragt', '\\bdeutlich', '\\bmassiv', '\\bmassig', '\\bgross', '\\buitgebreid', '\\bgevorderd', '\\bveel\\b', '\\bmatige?\\b', '\\bvolledig', '\\bμετρι', '\\bμεγαλ', '\\bεκτεταμεν', '\\bευμεγεθ', '\\bσοβαρ', '\\bπληρη', '\\bголям', '\\bизразен', '\\bзначим', '\\bумерен', '\\bобилен', '\\bпълн')
GRADE_HIGH = re.compile('grade?[ao]?\\s*(3|4|iii|iv)\\b|icrs grade (iii|iv|3|4)|stupnja iv|stupnja iii|\\bgrado (3|4)\\b|\\bgrad (3|4)\\b|\\bgrade (3|4)\\b')
DEGENERATIVE_MARROW = _rx('subchondral', 'subcondral', 'subkondral', 'supkondraln', 'subchondraln', 'υποχονδρι', 'υπαρθρικ', 'субхондрал', 'subchondrale?', 'subartikuler', '\\bcyst', '\\bquist', '\\bzyste\\b', '\\bcistic', 'reactive', 'reactivo', 'degenerative', 'degenerativ', 'reaktiv', '\\bcisti\\b')
TRAUMA = _rx('\\bbruise\\b', '\\bcontusion', '\\bkontuz', '\\btrauma', '\\bimpaction\\b', '\\bpivot shift\\b', '\\bkissing\\b', '\\bacute\\b', '\\bagudo\\b', '\\bakut', '\\bpivot kaymasi\\b', '\\bcontusion osseuse\\b', '\\bbone bruise\\b', '\\bbotcontusie\\b', '\\bконтузион', '\\bμωλωπ', '\\bkontuzij', '\\bimpaktcij', '\\bimpakcij', '\\bfall\\b', '\\binjury\\b', '\\bimpression\\b')
SYNOVIAL_PROXY = _rx('bursit', 'burzit', '\\bbursa\\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)', 'suprapatellar (bursitis|effusion|recess)', 'suprapatellar bursa', 'suprapatellar bursada', 'suprapatelarno', 'suprapatellaire recessus', 'hoffa', 'hoffit', 'plica', 'plika', 'πλικα', 'fat pad[^.]{0,20}(edema|oedema)', 'kapsul', 'capsul', 'καψ', 'капсул', '\\bpannus\\b', '\\bsinov', '\\bsynov')

def _polarity(clause: str, span=None) -> str:
    if UNCERTAIN.search(clause):
        return 'uncertain'
    if span is None or not FEATURES['directional_negation']:
        if NEGATION.search(clause):
            return 'negative'
    elif _negated(clause, span[0], span[1]):
        return 'negative'
    if NORMALITY.search(clause):
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return 'positive'
        return 'negative'
    return 'positive'

def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and (not low):
        return 1.0
    if low and (not high):
        return 0.45
    if high and low:
        return 0.8
    return 0.75

def _grade(n_pos, n_neg, n_unc, best):
    if n_pos or n_unc:
        score = min(0.97, 0.5 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.2 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = (0.28, 0.05)
    return (score, conf)

def _paired_weight(clause: str, meniscus: bool) -> float:
    g = _grade_of(clause) if FEATURES['graded_pathology'] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.3
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    elif tear:
        base = 1.0
    elif g is not None:
        base = 0.85 if g >= 2 else 0.3
    elif DEGEN.search(clause):
        base = 0.4
    else:
        base = 0.55
    if SEV_HIGH.search(clause) and (not SEV_LOW.search(clause)):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and (not SEV_HIGH.search(clause)):
        base *= 0.7
    return base

def _score_paired(cls, tgt):
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = 'Meniscus' in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == 'positive':
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return (s, cf, n_pos, n_neg)

def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and (not path_rx.search(c)):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == 'positive':
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.3)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return (s, c, n_pos, n_neg)

def _score_oa(cls):
    acc = {t: {'pos': 0, 'neg': 0, 'unc': 0, 'best': 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = (0, 0, 0.0)
    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append('Medial OA')
        if tf_lat:
            hits.append('Lateral OA')
        if pf:
            hits.append('PF OA')
        if not hits:
            if pol == 'positive':
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == 'negative':
                g_neg += 1
            continue
        for t in hits:
            if pol == 'positive':
                acc[t]['pos'] += 1
                acc[t]['best'] = max(acc[t]['best'], sev)
            elif pol == 'negative':
                acc[t]['neg'] += 1
            else:
                acc[t]['unc'] += 1
                acc[t]['best'] = max(acc[t]['best'], 0.3)
    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = (a['pos'], a['neg'], a['unc'], a['best'])
        if not (pos or unc) and g_pos and FEATURES['oa_inherit']:
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out

def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt in ('Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'):
        if tgt == 'Contusion':
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt), context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    if FEATURES['synovitis_backoff'] and out['Synovitis__npos'] == 0 and (out['Synovitis__nneg'] == 0):
        proxy = sum((1 for c in cls if SYNOVIAL_PROXY.search(c) and _polarity(c) == 'positive'))
        eff = out['Effusion']
        prior = 0.3 + 0.3 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out['Synovitis'] = min(0.72, prior)
        out['Synovitis__conf'] = 0.18
    return out

## Mount discovery and the safety-net submission

An all-0.5 `submission.csv` is written before anything else, so the run always leaves a scoreable file behind.

In [ ]:
import os
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
T0 = time.time()

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for d1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [d1] + sorted((p for p in d1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError('competition mount not found')
ROOT = find_root()
log(f'input root: {ROOT}')
_test_df = pd.read_csv(ROOT / 'test.csv')
_bench = _test_df[['StudyInstanceUID']].copy()
for _c in TARGETS:
    _bench[_c] = 0.5
_bench.to_csv('submission.csv', index=False)
log(f'benchmark submission.csv written ({len(_bench)} rows)')
STAGE_OK = {}

def stage(name):

    def deco(fn):

        def run(*a, **k):
            t = time.time()
            try:
                out = fn(*a, **k)
                STAGE_OK[name] = True
                log(f"stage '{name}' ok in {time.time() - t:.1f}s")
                return out
            except Exception:
                import traceback
                traceback.print_exc()
                STAGE_OK[name] = False
                log(f"stage '{name}' FAILED after {time.time() - t:.1f}s")
                return None
        return run
    return deco

In [ ]:
if RUN_EDA:
    train_df = pd.read_csv(ROOT / 'train.csv')
    log(f'train {train_df.shape}  test {_test_df.shape}')
    t = time.time()
    LAB = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    LAB['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    LAB = LAB.set_index('StudyInstanceUID')
    log(f'read {len(LAB)} reports in {time.time() - t:.1f}s')
    GOLD = train_df.dropna(subset=TARGETS).set_index('StudyInstanceUID')[TARGETS]
    log(f'{len(GOLD)} studies carry the twelve annotations')
    pos = (LAB[TARGETS] > 0.5).mean()
    sil = pd.Series({t_: float(((LAB[t_ + '__npos'] == 0) & (LAB[t_ + '__nneg'] == 0)).mean()) for t_ in TARGETS})
    print(pd.DataFrame({'derived positive rate': pos.round(3), 'silence rate': sil.round(3), 'annotated positive rate': GOLD.mean().round(3)}).to_string())

In [ ]:
if RUN_EDA:
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_auc_score
    plt.rcParams.update({'figure.dpi': 120, 'font.size': 8, 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})
    INK, ACC, WARN = ('#22303f', '#2b7a9b', '#c25a3d')

    def agreement(lab, n_boot=2000, seed=0):
        rng = np.random.default_rng(seed)
        g = lab.loc[GOLD.index]
        rows = []
        for t_ in TARGETS:
            y = GOLD[t_].values.astype(int)
            p = g[t_].values
            if len(set(y)) < 2:
                rows.append((t_, np.nan, np.nan, np.nan, int(y.sum()), int((1 - y).sum())))
                continue
            a = roc_auc_score(y, p)
            bs = []
            for _ in range(n_boot):
                i = rng.integers(0, len(y), len(y))
                if len(set(y[i])) > 1:
                    bs.append(roc_auc_score(y[i], p[i]))
            rows.append((t_, a, np.percentile(bs, 2.5), np.percentile(bs, 97.5), int(y.sum()), int((1 - y).sum())))
        return pd.DataFrame(rows, columns=['target', 'auc', 'lo', 'hi', 'npos', 'nneg'])
    AGREE = agreement(LAB).dropna(subset=['auc'])
    print(AGREE.round(3).to_string(index=False))
    print(f'\nmacro agreement AUC: {AGREE.auc.mean():.4f}   mean silence rate: {sil.mean() * 100:.1f}%')
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw={'width_ratios': [1.25, 1]})
    o = AGREE.sort_values('auc')
    y = np.arange(len(o))
    ax[0].hlines(y, o.lo, o.hi, color=ACC, lw=3, alpha=0.35)
    ax[0].plot(o.auc, y, 'o', color=ACC, ms=5)
    ax[0].axvline(0.5, color=INK, lw=0.8, ls=':')
    ax[0].axvline(o.auc.mean(), color=WARN, lw=1, ls='--')
    ax[0].text(o.auc.mean(), -0.9, f' macro {o.auc.mean():.3f}', color=WARN, fontsize=7)
    ax[0].set_ylim(-1.4, len(o) - 0.4)
    ax[0].set_yticks(y)
    ax[0].set_yticklabels([f'{t_}  ({p}+/{n}-)' for t_, p, n in zip(o.target, o.npos, o.nneg)])
    ax[0].set_xlim(0.35, 1.02)
    ax[0].set_xlabel('AUC of the derived score against the annotation')
    ax[0].set_title('gauge one: agreement, n = 58\nbars are 95% bootstrap intervals — they are this wide on purpose', loc='left', fontsize=8)
    o2 = sil.sort_values()
    ax[1].barh(np.arange(len(o2)), o2.values * 100, color=INK, alpha=0.8, height=0.65)
    ax[1].set_yticks(np.arange(len(o2)))
    ax[1].set_yticklabels(o2.index)
    ax[1].set_xlabel('% of studies where no rule fired at all')
    ax[1].set_title('gauge two: coverage, n = 4 407\nno labels needed, so it runs on the whole corpus', loc='left', fontsize=8)
    for i, v in enumerate(o2.values * 100):
        ax[1].text(v + 1, i, f'{v:.0f}', va='center', fontsize=6.5, color=INK)
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_EDA:
    _SCRIPT = {'el': re.compile('[Ͱ-Ͽ]'), 'bg/ru': re.compile('[Ѐ-ӿ]')}
    _STOP = {'en': '\\b(the|and|is|with|there is|normal)\\b', 'es': '\\b(del|los|las|con|sin|senal|rodilla|hallazgos|tecnica|resultados|impresion|menisco|rotura)\\b', 'fr': '\\b(des|les|avec|sans|genou|aucune)\\b', 'nl': '\\b(van|het|een|geen|met|voorste|knie)\\b', 'de': '\\b(der|die|und|mit|ohne|kein|keine|nachweis)\\b', 'tr': '\\b(ve|ile|izlenmistir|mevcut|normaldir|diz|bulgular)\\b', 'hr': '\\b(se|te|uz|bez|prikaz|uredan|koljena|meniska)\\b'}
    _STOP = {k: re.compile(v) for k, v in _STOP.items()}

    def guess_language(report):
        n = normalize(report)
        for tag, rx in _SCRIPT.items():
            if rx.search(n):
                return tag
        score = {k: len(rx.findall(n)) for k, rx in _STOP.items()}
        best = max(score, key=score.get)
        return best if score[best] >= 2 else '?'
    LANG = pd.Series([guess_language(r) for r in train_df['Report'].fillna('')], index=train_df['StudyInstanceUID'])
    print(LANG.value_counts().to_string())
    SIL = pd.DataFrame({t: ((LAB[t + '__npos'] == 0) & (LAB[t + '__nneg'] == 0)).values for t in TARGETS}, index=LAB.index)
    by_lang = SIL.groupby(LANG.reindex(SIL.index).values).mean() * 100
    by_lang = by_lang.loc[LANG.value_counts().index.intersection(by_lang.index)]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={'width_ratios': [1.7, 1]})
    im = ax[0].imshow(by_lang.values, cmap='RdYlBu_r', vmin=0, vmax=100, aspect='auto')
    ax[0].set_xticks(range(len(TARGETS)))
    ax[0].set_xticklabels(TARGETS, rotation=55, ha='right', fontsize=6.5)
    ax[0].set_yticks(range(len(by_lang)))
    ax[0].set_yticklabels([f'{l}  (n={int((LANG == l).sum())})' for l in by_lang.index], fontsize=7)
    ax[0].grid(False)
    for i in range(by_lang.shape[0]):
        for j in range(by_lang.shape[1]):
            v = by_lang.values[i, j]
            ax[0].text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=5.5, color='white' if v > 62 or v < 12 else INK)
    ax[0].set_title('gauge two, broken down: % of studies where no rule fired\na common finding silent in one language and not another is a lexicon gap — or a reporting style', loc='left', fontsize=8)
    fig.colorbar(im, ax=ax[0], fraction=0.02, pad=0.01)
    CLAUSES = {u: clauses(r) for u, r in zip(train_df['StudyInstanceUID'], train_df['Report'].fillna(''))}

    def names_it(uid, target):
        cs = CLAUSES[uid]
        if any((ANAT_MATCH[target].search(c) for c in cs)):
            return True
        if 'Meniscus' in target:
            return any((PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)) for c in cs))
        return False
    rows = []
    for t_ in ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']:
        sel = SIL[t_].values
        if not sel.sum():
            continue
        named = np.array([names_it(u, t_) for u in SIL.index[sel]])
        rows.append((t_, int(sel.sum()), 100 * named.mean()))
    D = pd.DataFrame(rows, columns=['target', 'silent', 'names it anyway'])
    y = np.arange(len(D))
    ax[1].barh(y, 100 - D['names it anyway'], color='#8a97a3', label='never mentioned')
    ax[1].barh(y, D['names it anyway'], left=100 - D['names it anyway'], color=WARN, label='mentioned, missed')
    ax[1].set_yticks(y)
    ax[1].set_yticklabels([f'{t_}\n({n} silent)' for t_, n in zip(D.target, D.silent)], fontsize=6.5)
    ax[1].set_xlabel('% of the silent studies')
    ax[1].legend(fontsize=6.5, frameon=False, loc='lower right')
    ax[1].set_title('why it was silent\nonly the orange half is a lexicon gap', loc='left', fontsize=8)
    fig.tight_layout()
    plt.show()
    print(D.round(1).to_string(index=False))

## Stage 1 — DINOv2 pool

Device probe, decoding rules, slot selection, the pixel cache, the model definition, and the weighted rank mean over every attached member.

In [ ]:
from __future__ import annotations
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback')
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device('cpu')]
print(f'devices: {[str(d) for d in DEVS]}')
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

In [ ]:
GPU_OK = any(str(_d).startswith('cuda') for _d in DEVS)
if not GPU_OK:
    print('\n' + '!' * 78)
    print('NO USABLE GPU. The probe above rejected every CUDA device, so stage 1 will')
    print('run on CPU and stages 2-3 will be skipped -- at full test-set size that')
    print('cannot finish inside 9 h. Kaggle\'s P100 (sm_60) has no compiled kernels in')
    print('this torch build and fails even on a plain tensor copy.')
    print('FIX: Session options -> Accelerator -> GPU T4 x2, then re-run.')
    print('!' * 78 + '\n', flush=True)

In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for depth1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [depth1] + sorted((p for p in depth1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError(f'competition mount not found (cwd {Path.cwd()}); expected a directory holding test.csv and test_series/')

def find_dinov2(variant='small'):
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 'config.json' in files and 'dinov2' in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None
LABEL_COLS = TARGETS + [t + '__conf' for t in TARGETS]

class LabelSourceError(RuntimeError):
    pass

def find_label_table():
    base = Path('/kaggle/input')
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
            cands += [Path(root) / f for f in files if f.startswith('report_labels') and f.endswith('.csv')]
    cands += [p for p in (Path('data/derived/report_labels_v2.csv'),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if 'StudyInstanceUID' in head.columns and all((t in head.columns for t in TARGETS)):
            return c
    return None

def label_mount_attached():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return False
    return any(('label' in p.name.lower() for p in base.iterdir() if p.is_dir()))

def read_labels(train_df):
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    lab['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    lab = lab.set_index('StudyInstanceUID')
    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError('LABEL SOURCE: a label dataset is mounted but no usable table was found in it. Falling back to the lexicon here would train on the weaker labels and say so only in a log line, so the run stops instead.')
        log(f'LABEL SOURCE: lexicon, {n} studies (no table mounted)')
        return lab
    tab = pd.read_csv(src).set_index('StudyInstanceUID')
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(f'LABEL SOURCE: {src} is missing {len(missing)} expected columns (first: {missing[0]!r}). Refusing to fall back silently.')
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(f'LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.')
    log(f'LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, lexicon for the remaining {n - len(hit)}')
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab
ROOT = find_root()
log(f'input root: {ROOT}')
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')

In [ ]:
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

In [ ]:
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def cache_tag(rules=None):
    r = dict(RULES if rules is None else rules)
    t = f'{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}'
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += '_' + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

In [ ]:
def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

In [ ]:
class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

In [ ]:
class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

In [ ]:
def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)

In [ ]:
FINGERPRINT_TOL = 0.002

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass

def find_weights(name='manifest.json'):
    import json
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if name not in files:
            continue
        try:
            man = json.loads((Path(root) / name).read_text())
        except (OSError, ValueError):
            continue
        if isinstance(man.get('members'), list) and man['members']:
            missing = [m['file'] for m in man['members'] if not (Path(root) / m['file']).is_file()]
            if missing:
                raise WeightsError(f"{root} holds a manifest listing {len(man['members'])} members but {len(missing)} of their files are absent (first {missing[0]!r})")
            return Path(root)
    return None
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
LEGACY_MEMBER_WEIGHT_BY_TARGET = {'Lateral Meniscus': 15.0, 'Medial OA': 2.5, 'Lateral OA': 15.0, 'Contusion': 5.0}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out = ([], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()
LEGACY_BUNDLE_FILE = 'rsna_20260807_v1.pt'
LEGACY_WEIGHT = 0.5

def find_legacy_bundle():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if LEGACY_BUNDLE_FILE in files:
            return Path(root) / LEGACY_BUNDLE_FILE
    return None

def legacy_group_members():
    p = find_legacy_bundle()
    if p is None:
        log('no legacy bundle attached; blending skipped')
        return {}
    try:
        b = torch.load(p, map_location='cpu', weights_only=False)
        folds = b.get('fold_states') or []
        b_slots = [tuple(s)[0] for s in b.get('slots', SLOTS)]
        if list(b.get('targets', TARGETS)) != TARGETS or b_slots != [s[0] for s in SLOTS]:
            log(f'legacy bundle {p.name}: target/slot contract differs; blending skipped')
            return {}
        gr, n_gr = (int(b.get('group', 3)), int(b.get('n_group', 3)))
        variant = str(b.get('model_variant', 'dinov2-small')).split('-')[-1]
        key = json.dumps({'img': int(b.get('img', 224)), 'group': gr, 'slices': gr * n_gr, 'crop_mm': 160.0, 'band': [0.2, 0.8], 'rules': RULES_LEGACY, 'slots': [s[0] for s in SLOTS]}, sort_keys=True)
        ms = [{'id': f"legacy-f{f.get('fold', k)}", 'fold': f.get('fold', k), 'state': f['state_dict'], 'holdout': None, 'weight': LEGACY_WEIGHT, 'target_weight': [LEGACY_MEMBER_WEIGHT_BY_TARGET.get(t, 0.0) for t in TARGETS], 'pixel_group': key, 'config': {'unfreeze_last': 6, 'variant': 'base' if variant == 'base' else 'small', 'pool': 'cls_mean_focal', 'prior': True}} for k, f in enumerate(folds)]
        if ms:
            active = sorted(set(LEGACY_MEMBER_WEIGHT_BY_TARGET.values()))
            log(f'legacy bundle {p.name}: {len(ms)} fold(s) join with target-specific per-member weights {active}')
        return {key: ms} if ms else {}
    except Exception as exc:
        log(f'legacy bundle unusable ({type(exc).__name__}: {exc}); blending skipped')
        return {}

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            state, fp = (m['state'], None)
        else:
            ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
            state, fp = (ck['model'], ck.get('fingerprint'))
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p = predicted
    else:
        p, public_p = (predicted, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None):
        if float(np.std(pred)) < 1e-09:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'ids': ids, 'pred': public_pred})
            elif public_pred is not None:
                log(f"  {m['id']}: public-frontier vote omitted because only {len(starts)} / {len(starts_full)} windows completed")
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, 'submission.csv')
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s){(', jitter' if jitter else '')}); submission.csv = weighted rank mean of {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))
                starts, jit = (starts_full, False)
                if est['fixed'] is not None and est['win'] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est['fixed'] + est['win'] > room:
                        log(f'  {left / 60:.0f} min left: surrendering {len(pending)} member(s); not one more fits')
                        pending.clear()
                        return (None, None, False)
                    jit = est['fixed'] + 2 * len(starts_full) * est['win'] <= room * 0.6
                    per_win = est['win'] * (2 if jit else 1)
                    n_win = int((room - est['fixed']) / per_win) if per_win > 0 else len(starts_full)
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return (pending.pop(0), starts, jit)

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, (fs, ws) = _run_member(path, m, d, Cte, Mte, idx, starts, jit)
                        with STATE_LOCK:
                            est['fixed'], est['win'] = (fs, ws)
                        bank(m, st_te, p, starts, jit, public_p)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} ({type(exc).__name__}: {exc}); " + ('retrying on peer device' if attempt == 0 and others else 'dropped -- costs one vote, not the run'))
                        if d.type == 'cuda':
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        del Cte, Mte
        gc.collect()
    if not per_member:
        raise WeightsError('no member produced predictions; submission stays at 0.5')
    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, 'submission.csv')
    log(f'final submission.csv = weighted rank mean of {len(per_member)} member(s); {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(frontier_acc, frontier_ids, test_df, 'submission_public_0899.csv')
        log(f'submission_public_0899.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {frontier_sub.shape}; nulls {int(frontier_sub[TARGETS].isna().sum().sum())}')
    else:
        log(f'public-frontier fallback not emitted: {len(public_frontier_members)} / {len(members)} required public members completed')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")

In [ ]:
def take_group(cache_rows, g):
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]

def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

@torch.no_grad()
def predict(model, cache, mask, idx, dev, img_size=None):
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, g * GROUP:(g + 1) * GROUP])).to(dev)
            with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                z = model(rows, m, img_size).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)

def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j]) if len(set(y[:, j])) > 1 else np.nan for j in range(y.shape[1])]))

In [ ]:
def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def write_benchmark_submission():
    t = pd.read_csv(ROOT / 'test.csv')
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv('submission.csv', index=False)

def _v37_validate_submission(path, test_df, tag):
    path = Path(path)
    frame = pd.read_csv(path)
    expected = ['StudyInstanceUID'] + TARGETS
    if list(frame.columns) != expected:
        raise ValueError(f'{tag}: columns differ from the competition contract')
    if len(frame) != len(test_df) or not frame['StudyInstanceUID'].is_unique:
        raise ValueError(f'{tag}: row count or StudyInstanceUID uniqueness failed')
    if set(frame['StudyInstanceUID'].astype(str)) != set(test_df['StudyInstanceUID'].astype(str)):
        raise ValueError(f'{tag}: StudyInstanceUID set differs from test.csv')
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f'{tag}: non-finite prediction')
    return test_df[['StudyInstanceUID']].merge(frame, on='StudyInstanceUID', how='left')

def main():
    write_benchmark_submission()
    pkg = find_weights()
    if pkg is None and REQUIRE_WEIGHTS:
        raise WeightsError(
            'No weights package found under /kaggle/input. Attach '
            'pilkwang/rsna-knee-weights (it carries manifest.json + 20 m_*.pt). '
            'Set REQUIRE_WEIGHTS = False only if you really want the 9-hour '
            'from-scratch training fallback instead.')
    if pkg is not None:
        dev = DEVS[0]
        infer_from_package(pkg, dev)
        try:
            test_df = pd.read_csv(ROOT / 'test.csv')
            native_path = Path('submission.csv')
            public_path = Path('submission_public_0899.csv')
            native = _v37_validate_submission(native_path, test_df, 'native 24-member')
            public = _v37_validate_submission(public_path, test_df, 'public DINO frontier')
            native.to_csv('submission_native_pool.csv', index=False)
            if PROMOTE_PUBLIC_FRONTIER:
                public.to_csv(native_path, index=False)
                promoted = _v37_validate_submission(native_path, test_df, 'promoted primary')
                if not promoted.equals(public):
                    raise AssertionError('serialization differs from the validated public frontier')
                log('stage 1 primary = public-frontier pool (20 members, no jitter); '
                    'native pool kept at submission_native_pool.csv')
            else:
                log('stage 1 primary = native pool (jitter TTA, legacy folds if attached); '
                    'public frontier kept at submission_public_0899.csv')
        except Exception as public_frontier_error:
            log(f'public-frontier promotion skipped safely: {public_frontier_error}')
            traceback.print_exc()
        log('done')
        return
    read_labels(pd.read_csv(ROOT / 'train.csv', usecols=['StudyInstanceUID', 'Report']))
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    train_df = pd.read_csv(ROOT / 'train.csv')
    train_series = pd.read_csv(ROOT / 'train_series.csv')
    log(f'train {train_df.shape} test {test_df.shape}')
    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both['SeriesInstanceUID'], both['Anatomical_Plane']))
    log('header pass: test')
    hte = annotate(walk('test_series'))
    log(f'  {len(hte)} test series')
    log('header pass: train')
    htr = annotate(walk('train_series'))
    log(f'  {len(htr)} train series')
    slots_te, slots_tr = (pick_slots(hte, plane_map), pick_slots(htr, plane_map))
    cov = pd.Series([len(v) for v in slots_tr.values()]).describe()
    log(f"train slots per study: mean {cov['mean']:.2f} min {cov['min']:.0f} max {cov['max']:.0f}")
    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_of(htr, 'train '), 'train')
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_of(hte, 'test '), 'test')
    t_lab = time.time()
    lab = read_labels(train_df)
    log(f'derived labels for {len(lab)} studies in {time.time() - t_lab:.1f}s')
    gold = train_df.set_index('StudyInstanceUID')[TARGETS]
    gold = gold[gold.notna().all(axis=1)]
    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = (gold.loc[st].values, 3.0)
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            W[i] = 0.25 + 0.75 * r[[t + '__conf' for t in TARGETS]].values
    keep = np.where(W.sum(1) > 0)[0]
    log(f'supervised {len(keep)} of {len(st_tr)} studies (annotated {len(gold)})')
    import hashlib
    rep = train_df.set_index('StudyInstanceUID')['Report'].fillna('')
    grp = np.array([int(hashlib.md5(rep.get(s, s).encode()).hexdigest()[:8], 16) % 5 for s in st_tr])
    va = np.array([i for i in keep if grp[i] == 0])
    tr = np.array([i for i in keep if grp[i] != 0])
    if len(va) == 0 or len(tr) < BATCH_STUDIES:
        cut = max(1, len(keep) // 5)
        va, tr = (keep[:cut], keep[cut:])
    log(f'train {len(tr)} / holdout {len(va)} studies')
    gpos = {s: i for i, s in enumerate(st_tr)}
    va_set = set(va.tolist())
    gi = np.array([gpos[s] for s in gold.index if s in gpos and gpos[s] in va_set])
    gold_y = gold.loc[[st_tr[i] for i in gi]].values.astype(int) if len(gi) else None
    yv = (Y[va] > 0.5).astype(int)
    log(f'annotation check: {len(gi)} of {len(gold)} annotated studies are in the holdout')
    dev = DEVS[0]
    results, test_preds = ({}, {})
    for cfg in RUNS:
        pitch = CROP_MM / cfg['img']
        log(f"=== {cfg['name']}: {cfg['img']} px, {pitch:.3f} mm/pixel, {pitch * 14:.2f} mm per patch token ===")
        torch.manual_seed(SEED)
        model = build_model(UNFREEZE_LAST).to(dev)
        opt = torch.optim.AdamW([{'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': LR_BACKBONE}, {'params': model.head.parameters(), 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
        steps = max(EPOCHS * (len(tr) // BATCH_STUDIES), 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.15)
        scaler = torch.amp.GradScaler('cuda', enabled=dev.type == 'cuda')
        best, best_state, best_annot = (-1.0, None, float('nan'))
        for ep in range(EPOCHS):
            model.train()
            perm = np.random.permutation(tr)
            tot, nstep = (0.0, 0)
            for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
                sel = perm[b:b + BATCH_STUDIES]
                rows = torch.from_numpy(Ctr[sel]).to(dev)
                g = int(torch.randint(N_GROUP, (1,)).item())
                imgs = augment(take_group(rows, g))
                m = torch.from_numpy(Mtr[sel]).to(dev)
                y = torch.from_numpy(Y[sel]).to(dev)
                w = torch.from_numpy(W[sel]).to(dev)
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    loss = (F.binary_cross_entropy_with_logits(model(imgs, m, cfg['img']), y, reduction='none') * w).mean()
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                tot += loss.item()
                nstep += 1
            pv = predict(model, Ctr, Mtr, va, dev, cfg['img'])
            d = macro_auc(yv, pv)
            g_auc = float('nan')
            if gold_y is not None and len(gi):
                g_auc = macro_auc(gold_y, predict(model, Ctr, Mtr, gi, dev, cfg['img']))
            log(f'  epoch {ep + 1}/{EPOCHS}  loss {tot / max(nstep, 1):.4f}  holdout {d:.4f}  annot(n={len(gi)}) {g_auc:.4f}')
            if d > best:
                best, best_annot = (d, g_auc)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if time.time() - T0 > TIME_BUDGET:
                log('  time budget reached')
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        results[cfg['name']] = (best, best_annot)
        test_preds[cfg['name']] = predict(model, Cte, Mte, np.arange(len(st_te)), dev, cfg['img'])
        log(f"  {cfg['name']}: best holdout {best:.4f} (annot {best_annot:.4f})")
        del model, opt, sched, scaler, best_state
        gc.collect()
        if dev.type == 'cuda':
            torch.cuda.empty_cache()
    log('---- summary ----')
    for n, (d, g_auc) in results.items():
        log(f'  {n:12s} holdout {d:.4f}   annot {g_auc:.4f}')
    pick = max(results, key=lambda k: results[k][0])
    log(f'best on the holdout: {pick} ({results[pick][0]:.4f})')
    for name, pred in test_preds.items():
        sub = write_submission(pred, st_te, test_df, f'submission_{name}.csv')
        log(f'  submission_{name}.csv {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    ens = np.mean([pd.DataFrame(p).rank(pct=True).values for p in test_preds.values()], axis=0)
    write_submission(ens, st_te, test_df, 'submission_rankmean.csv')
    log(f'  submission_rankmean.csv (rank mean of {len(test_preds)})')
    sub = write_submission(test_preds[pick], st_te, test_df, 'submission.csv')
    log(f'submission.csv = {pick}; {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    print(sub.head().to_string())

In [ ]:
try:
    main()
except LabelSourceError:
    traceback.print_exc()
    raise
except Exception:
    traceback.print_exc()
    t = pd.read_csv(find_root() / 'test.csv')
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv('submission.csv', index=False)
    print('wrote fallback submission.csv')
log('done')

In [ ]:
# ---- stage 1 checkpoint -------------------------------------------------
import pandas as _s1_pd

_s1 = _s1_pd.read_csv('/kaggle/working/submission.csv')
wall_log(f'stage 1 complete: {_s1.shape[0]} rows, {_s1[TARGETS].isna().sum().sum()} nulls')
del _s1

## Stage 2 — DINOv3 folds

Five `vit_small_patch16_dinov3` folds, rank-blended in at `A5_W`. Skipped if the wall clock has already passed `DINOV3_START_CUTOFF_S`.

In [ ]:
# timm on the Kaggle image may predate the DINOv3 model registrations. Probe in a
# subprocess (so timm is not imported into this session yet) and, if the name is
# missing, install the wheel bundled with the fold-weights dataset.
import glob as _tg, subprocess as _tsp, sys as _tsys

_probe = _tsp.run(
    [_tsys.executable, '-c',
     "import timm; print('vit_small_patch16_dinov3' in set(timm.list_models()))"],
    capture_output=True, text=True)
_have_dinov3 = _probe.stdout.strip().splitlines()[-1:] == ['True']
print('timm registers vit_small_patch16_dinov3:', _have_dinov3)

if not _have_dinov3:
    try:
        _whl = []
        for _pat in ('/kaggle/input/*/timm-*.whl', '/kaggle/input/*/*/timm-*.whl',
                     '/kaggle/input/*/*/*/timm-*.whl'):
            _whl += _tg.glob(_pat)
        if not _whl:
            raise RuntimeError(
                'timm has no DINOv3 models and no timm wheel is attached; '
                'mattiaangeli/knee-mri-fold-weights ships timm-1.0.22-py3-none-any.whl')
        print('installing', _whl[0])
        _tsp.run([_tsys.executable, '-m', 'pip', 'install', '--no-deps', '--no-index',
                  '-q', _whl[0]], check=True)
        _probe = _tsp.run(
            [_tsys.executable, '-c',
             "import timm; print('vit_small_patch16_dinov3' in set(timm.list_models()))"],
            capture_output=True, text=True)
        _have_dinov3 = _probe.stdout.strip().splitlines()[-1:] == ['True']
        print('timm upgraded:', _have_dinov3)
    except Exception as _terr:
        print('timm upgrade failed:', type(_terr).__name__, _terr)
        _have_dinov3 = False

RUN_DINOV3 = (_have_dinov3
              and GPU_OK
              and (time.time() - WALL_T0) < DINOV3_START_CUTOFF_S
              and wall_left() > 600)
if not RUN_DINOV3:
    wall_log('SKIPPING stage 2 (DINOv3) -- stage 1 output stands. Cause: '
             + ('no usable GPU' if not GPU_OK
                else 'timm lacks the DINOv3 models' if not _have_dinov3
                else 'wall-clock budget spent'))

In [ ]:
if RUN_DINOV3:
    try:
        _A5_SAVED = dict(globals())
        import gc, os, time, warnings
        from concurrent.futures import ProcessPoolExecutor, as_completed
        from pathlib import Path
        import cv2
        import numpy as np
        import pandas as pd
        import pydicom
        import timm
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        warnings.filterwarnings('ignore')
        cv2.setNumThreads(1)
        CROP_MM = 130.0
        SIZE = 336
        SLICE_BAND = (0.12, 0.88)
        N_SLICE = 16
        INTENSITY = 'slice'
        SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
        N_SLOT = len(SLOTS)
        LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

        def _find_dir(*names):
            root = Path('/kaggle/input')
            cand = []
            for n in names:
                cand += [root / n, root / 'competitions' / n, root / 'datasets' / n]
                for parent in (root / 'datasets', root / 'competitions', root):
                    if parent.is_dir():
                        try:
                            cand += [d / n for d in parent.iterdir() if d.is_dir()]
                        except OSError:
                            pass
            for p in cand:
                if p.is_dir():
                    return p
            return None
        COMP = _find_dir('rsna-knee-abnormality-detection')
        CKPT = _find_dir('knee-mri-fold-weights')
        assert COMP is not None, 'competition data not attached'
        assert CKPT is not None, 'fold weights not attached'
        assert (COMP / 'sample_submission.csv').exists(), f'no competition data at {COMP}'
        assert list(CKPT.glob('*_f*.pt')), f'no checkpoints at {CKPT}'
        DEV = str(DEVS[0])  # probed and verified in stage 1, not merely is_available()
        print(f'competition : {COMP}')
        print(f'checkpoints : {CKPT}')
        print(f'device      : {DEV}')
        for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
            cc = torch.cuda.get_device_capability(i)
            print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')
    except Exception:
        import traceback as _s2_tb
        _s2_tb.print_exc()
        # Restore first: _A5_SAVED was captured while RUN_DINOV3 was still True,
        # so replaying it after the assignment would switch the stage back on.
        for _k, _v in globals().get('_A5_SAVED', {}).items():
            globals()[_k] = _v
        RUN_DINOV3 = False
        wall_log('stage 2 FAILED -- keeping the stage 1 submission; stage 3 still runs')

In [ ]:
if RUN_DINOV3:
    try:
        SERIES_ROOT = COMP / 'test_series'
        if not SERIES_ROOT.exists():
            SERIES_ROOT = COMP / 'train_series'
        print('series root:', SERIES_ROOT)

        def ordered_files(sdir, cap=64):
            keyed = []
            for f in sdir.glob('*.dcm'):
                try:
                    ds = pydicom.dcmread(str(f), stop_before_pixels=True)
                    keyed.append((int(ds.InstanceNumber), str(f)))
                except Exception:
                    continue
                if len(keyed) >= cap * 4:
                    break
            return [f for _, f in sorted(keyed)]

        def series_side(path):
            try:
                return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
            except Exception:
                return 0.0

        def read_crop(path):
            try:
                ds = pydicom.dcmread(path)
                arr = ds.pixel_array.astype(np.float32)
            except Exception:
                return None
            try:
                ps = float(ds.PixelSpacing[0])
            except Exception:
                ps = CROP_MM / max(arr.shape)
            half = int(round(CROP_MM / ps / 2))
            cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
            y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
            x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
            crop = arr[y0:y1, x0:x1]
            return None if crop.size == 0 else crop

        def window(crop, lo, hi, flip):
            c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
            img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
            return img[:, ::-1].copy() if flip else img

        def render(path, flip):
            crop = read_crop(path)
            if crop is None:
                return None
            lo, hi = np.percentile(crop[::4, ::4], [1, 99])
            return window(crop, lo, hi, flip)

        def build_study(args):
            idx, study, recs = args
            out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
            mask = np.zeros(N_SLOT, np.uint8)
            rows = pd.DataFrame(recs)
            if len(rows):
                for s_i, (plane, fs) in enumerate(SLOTS):
                    sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
                    if sub.empty:
                        continue
                    files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
                    if not files:
                        continue
                    flip = plane != 'Sagittal' and series_side(files[0]) < 0
                    lo, hi = SLICE_BAND
                    i0 = int(round(lo * (len(files) - 1)))
                    i1 = int(round(hi * (len(files) - 1)))
                    avail = list(range(i0, i1 + 1))
                    if len(avail) >= N_SLICE:
                        picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                        off = 0
                    else:
                        picks, off = (avail, (N_SLICE - len(avail)) // 2)
                    if INTENSITY == 'series':
                        crops = [read_crop(files[p]) for p in picks]
                        got = [x for x in crops if x is not None]
                        if got:
                            samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                            lo_, hi_ = np.percentile(samp, [1, 99])
                            for c, x in enumerate(crops):
                                if x is None:
                                    x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                                if x is not None:
                                    out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
                    else:
                        for c, p in enumerate(picks):
                            img = render(files[p], flip)
                            if img is None:
                                img = render(files[min(len(files) - 1, p + 1)], flip)
                            if img is not None:
                                out[s_i, off + c] = (img * 255).astype(np.uint8)
                    mask[s_i] = len(picks)
            return (idx, out, mask)
        sub_df = pd.read_csv(COMP / 'sample_submission.csv')
        ser_csv = pd.read_csv(COMP / 'test_series.csv')
        if not (COMP / 'test_series').exists():
            ser_csv = pd.read_csv(COMP / 'train_series.csv')
        ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
        studies = sub_df.StudyInstanceUID.tolist()
        by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
        print(f'{len(studies):,} test studies, {len(by):,} with series metadata')
    except Exception:
        import traceback as _s2_tb
        _s2_tb.print_exc()
        # Restore first: _A5_SAVED was captured while RUN_DINOV3 was still True,
        # so replaying it after the assignment would switch the stage back on.
        for _k, _v in globals().get('_A5_SAVED', {}).items():
            globals()[_k] = _v
        RUN_DINOV3 = False
        wall_log('stage 2 FAILED -- keeping the stage 1 submission; stage 3 still runs')

In [ ]:
if RUN_DINOV3:
    try:
        N_SLOT_TYPES, MASK_IDX = (6, 0)

        def segment_softmax(scores, sidx, B):
            T, K = scores.shape
            idx = sidx.unsqueeze(1).expand(-1, K)
            m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
            m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
            e = (scores - m[sidx]).exp()
            s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
            return e / s[sidx].clamp(min=1e-06)

        class MeanMaxPool(nn.Module):

            def forward(self, f, sidx, B, slot=None, return_attn=False):
                D = f.shape[1]
                cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
                mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
                mean = mean / cnt.clamp(min=1).unsqueeze(1)
                mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
                mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
                return (torch.cat([mean, mx], 1), None)

        class LabelAttentionPool(nn.Module):

            def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
                super().__init__()
                self.d, self.k, self.h = (d, n_labels, n_heads)
                self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
                self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
                self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

            def forward(self, f, sidx, B, slot=None, return_attn=False):
                scores = self.key(f) @ self.q.t() / self.d ** 0.5
                if self.slot_bias is not None and slot is not None:
                    scores = scores + self.slot_bias.t()[slot]
                a = segment_softmax(scores, sidx, B)
                out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
                out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
                return (out, a)

        class TokenXAttnPool(nn.Module):

            def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
                super().__init__()
                self.d, self.k = (d, n_labels)
                self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
                self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
                self.kv_norm = nn.LayerNorm(d)
                self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

            def forward(self, tok, sidx, B, slot=None, return_attn=False):
                T, N, D = tok.shape
                cnt = torch.bincount(sidx, minlength=B)
                S = int(cnt.max().item())
                starts = torch.cumsum(cnt, 0) - cnt
                pos = torch.arange(T, device=tok.device) - starts[sidx]
                kv = tok + self.slot_emb(slot).unsqueeze(1)
                pad = tok.new_zeros(B, S, N, D)
                pad[sidx, pos] = kv
                keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
                keep[sidx, pos] = True
                kpm = ~keep.repeat_interleave(N, dim=1)
                pad = self.kv_norm(pad.reshape(B, S * N, D))
                q = self.q.unsqueeze(0).expand(B, -1, -1)
                att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
                cls = tok[:, 0]
                mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
                mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
                mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
                base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
                return (torch.cat([att, base], -1), w)

        class ViTSlotToken(nn.Module):

            def __init__(self, vit, n_cat, dim=None):
                super().__init__()
                self.vit = vit
                d = dim or vit.embed_dim
                self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
                self.num_features = vit.num_features
                self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
                vit.num_prefix_tokens = self._orig_prefix + 1
                for blk in vit.blocks:
                    a = getattr(blk, 'attn', None)
                    if a is not None and hasattr(a, 'num_prefix_tokens'):
                        a.num_prefix_tokens = a.num_prefix_tokens + 1

            @staticmethod
            def _maybe(mod, x):
                return x if mod is None else mod(x)

            def forward_features(self, x, cat):
                v = self.vit
                x = v.patch_embed(x)
                pos = v._pos_embed(x)
                rope = None
                if isinstance(pos, tuple):
                    x, rope = pos
                else:
                    x = pos
                x = self._maybe(getattr(v, 'patch_drop', None), x)
                x = self._maybe(getattr(v, 'norm_pre', None), x)
                npt = self._orig_prefix
                tok = self.tok(cat).unsqueeze(1)
                x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
                if rope is not None:
                    if getattr(v, 'rope_mixed', False):
                        for i, blk in enumerate(v.blocks):
                            x = blk(x, rope=rope[i])
                    else:
                        for blk in v.blocks:
                            x = blk(x, rope=rope)
                else:
                    x = v.blocks(x)
                return v.norm(x)

            def forward_head(self, x, pre_logits=True):
                return self.vit.forward_head(x, pre_logits=pre_logits)
        IMAGENET_MEAN = (0.485, 0.456, 0.406)
        IMAGENET_STD = (0.229, 0.224, 0.225)

        class _GatedDepthBlock(nn.Module):

            def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
                super().__init__()
                self.norm = nn.GroupNorm(1, n_slice)
                self.v = nn.Conv2d(n_slice, n_slice, 1)
                self.g = nn.Conv2d(n_slice, n_slice, 1)
                self.out = nn.Conv2d(n_slice, n_slice, 1)
                self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
                self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

            def forward(self, x):
                z = self.norm(x)
                return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

        class DepthCompress(nn.Module):

            def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
                super().__init__()
                self.imagenet = imagenet
                self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
                self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
                if imagenet:
                    self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
                    self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

            def forward(self, x):
                keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
                z = x
                for b in self.blocks:
                    z = b(z)
                z = self.proj(z)
                if self.imagenet:
                    z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
                return z * keep
        N_PLANE, N_CONTRAST = (3, 2)
        _PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
        _CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

        class SlotDepthMixer(nn.Module):

            def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
                super().__init__()
                self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
                self.alpha_max = alpha_max
                b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
                self.register_buffer('base', b.log()[self.r:])
                n_u = self.r + 1
                self.shared = nn.Parameter(torch.zeros(n_u))
                self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
                self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
                self.g0 = nn.Parameter(torch.zeros(()))
                self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
                self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
                idx = torch.arange(n_slice)
                self.register_buffer('off', idx[None, :] - idx[:, None])

            def kernel(self, slot):
                p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
                half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
                full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
                return F.softmax(full, dim=-1)

            def alpha(self, slot):
                p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
                return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

            def forward(self, x, slot, vmask):
                T, S, H, W = x.shape
                if vmask is None:
                    raise ValueError('stem=mixer requires the padding mask')
                k = self.kernel(slot)
                v = vmask.to(k.dtype)
                d = self.off + self.r
                inb = (d >= 0) & (d < self.ksize)
                kk = k[:, d.clamp(0, self.ksize - 1)] * inb
                M = kk * v[:, None, :]
                den = M.sum(-1, keepdim=True)
                eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
                ok = (den > 1e-06) & v[:, :, None].bool()
                M = torch.where(ok, M / den.clamp(min=1e-06), eye)
                a = self.alpha(slot)[:, None, None]
                Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
                if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
                    y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
                    return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
                return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

        def _seg_mean_max(v, sidx, B):
            D = v.shape[1]
            cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
            mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
            mean = mean / cnt.clamp(min=1).unsqueeze(1)
            mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
            mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
            return torch.cat([mean, mx], 1)

        def _pad_kv(x, sidx, B, norm):
            T, P, D = x.shape
            cnt = torch.bincount(sidx, minlength=B)
            S = int(cnt.max().item())
            starts = torch.cumsum(cnt, 0) - cnt
            pos = torch.arange(T, device=x.device) - starts[sidx]
            pad = x.new_zeros(B, S, P, D)
            pad[sidx, pos] = x
            keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
            keep[sidx, pos] = True
            return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

        class _GatedDelta(nn.Module):

            def __init__(self, d, n_labels, n_heads, dropout):
                super().__init__()
                self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
                self.kv_norm = nn.LayerNorm(d)
                self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
                self.d_norm = nn.LayerNorm(d)
                self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
                self.db = nn.Parameter(torch.zeros(n_labels))
                self.gate = nn.Parameter(torch.zeros(n_labels))

            def delta(self, pat, sidx, B, return_attn):
                kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
                q = self.q.unsqueeze(0).expand(B, -1, -1)
                att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
                return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

        class TokenResidualPool(_GatedDelta):

            def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
                super().__init__(d, n_labels, n_heads, dropout)
                self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

            def forward(self, tok, slot, sidx, B, pres, return_attn=False):
                base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
                d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
                return (base + self.gate * d_, w)

        class CodexResidualPool(_GatedDelta):

            def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
                super().__init__(d, n_labels, n_heads, dropout)
                self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

            def forward(self, tok, slot, sidx, B, pres, return_attn=False):
                base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
                d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
                return (base + self.gate * d_, w)

        class ClsAddPool(nn.Module):

            def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
                super().__init__()
                self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

            def forward(self, tok, slot, sidx, B, pres, return_attn=False):
                return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

        class Readout(nn.Module):

            def __init__(self, pool, d, n_labels=12, pe=64):
                super().__init__()
                self.pool_kind, self.k = (pool, n_labels)
                self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
                if pool in ('xres', 'clsadd', 'xcodex'):
                    self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
                elif pool in ('attn', 'xattn'):
                    if pool == 'xattn':
                        self.pool = TokenXAttnPool(d, n_labels)
                        wd = 3 * d + pe
                    else:
                        self.pool = LabelAttentionPool(d, n_labels)
                        wd = d + pe
                    self.norm = nn.LayerNorm(wd)
                    self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
                    self.b = nn.Parameter(torch.zeros(n_labels))
                else:
                    self.pool = MeanMaxPool()
                    self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
                self.drop = nn.Dropout(0.2)

            def forward(self, f, slot, sidx, B, return_attn=False):
                pe = self.pres_emb(slot)
                pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
                if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
                    return self.pool(f, slot, sidx, B, pres)[0]
                pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
                if self.pool_kind in ('attn', 'xattn'):
                    x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
                    x = self.drop(self.norm(x))
                    return (x * self.w).sum(-1) + self.b
                return self.net(torch.cat([pooled, pres], 1))

        class Net(nn.Module):

            def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
                super().__init__()
                self.enc, self.cond = (enc, cond)
                self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
                self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
                self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
                D = enc.num_features
                self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
                self.readout = Readout(pool, D)
                if cond == 'post':
                    self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

            def forward(self, im, slot, smeta, sidx, B, vm=None):
                if self.mixer is not None:
                    im = self.mixer(im, slot, vm)
                if self.compress is not None:
                    im = self.compress(im)
                f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
                if self.tokens:
                    inner = getattr(self.enc, 'vit', self.enc)
                    orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
                    f = torch.cat([f[:, :1], f[:, orig:]], 1)
                else:
                    f = self.enc.forward_head(f, pre_logits=True)
                    if f.dim() > 2:
                        f = f.flatten(1)
                ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
                if self.cond == 'post':
                    f = f + ex(self.slot_emb(slot))
                if self.meta_mlp is not None and smeta.shape[1] > 0:
                    mt = self.meta_mlp(smeta)
                    f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
                return self.readout(f, slot, sidx, B)
        models = []
        for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
            z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            cfg = z['cfg']
            _stem = cfg.get('stem', 'native')
            _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
            enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
            if cfg['cond'] == 'token':
                enc = ViTSlotToken(enc, N_SLOT_TYPES)
            m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
            missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
            assert not [k for k in missing if not k.startswith('enc.')], f'missing {missing[:5]}'
            assert not unexpected, f'unexpected {unexpected[:5]}'
            models.append(m.eval())
            print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
        CFG = cfg
        assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
        print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")
    except Exception:
        import traceback as _s2_tb
        _s2_tb.print_exc()
        # Restore first: _A5_SAVED was captured while RUN_DINOV3 was still True,
        # so replaying it after the assignment would switch the stage back on.
        for _k, _v in globals().get('_A5_SAVED', {}).items():
            globals()[_k] = _v
        RUN_DINOV3 = False
        wall_log('stage 2 FAILED -- keeping the stage 1 submission; stage 3 still runs')

In [ ]:
if RUN_DINOV3:
    try:
        AMP_PREF = 'bf16'

        def amp_for(dev):
            if not str(dev).startswith('cuda'):
                return (torch.float32, False)
            cc = torch.cuda.get_device_capability(dev)
            if AMP_PREF == 'bf16':
                return (torch.bfloat16, True)
            if AMP_PREF == 'fp16':
                return (torch.float16, True)
            if AMP_PREF == 'fp32':
                return (torch.float32, False)
            return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
        AMP_DT, AMP_ON = amp_for(DEV)
        WORKERS = max(1, min(4, os.cpu_count() or 4))
        CHUNK = 48
        MICRO = 8
        models = [m.to(DEV).eval() for m in models]
        print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

        def _norm_(im):
            k = CFG.get('norm', 'none')
            if k == 'zscore':
                m = (im > 0).float()
                n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
                mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
                var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
                return (im - mu) / (var.sqrt() + 1e-06) * m
            if k == 'imagenet':
                m = (im > 0).float()
                return (im - 0.485) / 0.229 * m
            return im

        @torch.no_grad()
        def _micro(images, masks):
            dev = DEV
            ims, slots, sidx, vms = ([], [], [], [])
            for b in range(len(masks)):
                present = np.nonzero(masks[b] > 0)[0]
                if len(present) == 0:
                    continue
                blk = images[b][present]
                ims.append(torch.from_numpy(blk))
                vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
                slots.append(torch.from_numpy(present + 1).long())
                sidx.append(torch.full((len(present),), b, dtype=torch.long))
            out = np.full((len(masks), len(LABELS)), np.nan, np.float32)
            if not ims:
                return out
            im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
            sl = torch.cat(slots).to(dev)
            si = torch.cat(sidx).to(dev)
            vm = torch.cat(vms).to(dev)
            sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
            acc = torch.zeros(len(masks), len(LABELS), device=dev, dtype=torch.float32)
            with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu', dtype=AMP_DT, enabled=AMP_ON):
                for m in models:
                    acc += torch.sigmoid(m(im, sl, sm, si, len(masks), vm=vm).float())
            got = (acc / len(models)).cpu().numpy()
            keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])
            out[keep] = got[keep]
            return out

        def predict(images, masks):
            out = np.full((len(masks), len(LABELS)), np.nan, np.float32)
            for a in range(0, len(masks), MICRO):
                b = min(a + MICRO, len(masks))
                out[a:b] = _micro(images[a:b], masks[a:b])
            return out
        preds = np.full((len(studies), len(LABELS)), np.nan, np.float32)
        t0, done = (time.time(), 0)
        with ProcessPoolExecutor(max_workers=WORKERS) as ex:
            for c0 in range(0, len(studies), CHUNK):
                block = studies[c0:c0 + CHUNK]
                imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
                msks = np.zeros((len(block), N_SLOT), np.uint8)
                futs = [ex.submit(build_study, (i, s, by.get(s, []))) for i, s in enumerate(block)]
                for f in as_completed(futs):
                    try:
                        i, a, k = f.result()
                        imgs[i], msks[i] = (a, k)
                    except Exception as e:
                        print(f'  study failed: {type(e).__name__}: {e}')
                preds[c0:c0 + len(block)] = predict(imgs, msks)
                done += len(block)
                el = time.time() - t0
                print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
                del imgs, msks
                gc.collect()
        print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
        A5_W = 0.45
        A5_LABELS = list(LABELS)
        A5_PREDS = dict(zip(sub_df['StudyInstanceUID'].astype(str), preds))
        for _a5k, _a5v in _A5_SAVED.items():
            globals()[_a5k] = _a5v
        del _A5_SAVED, _a5k, _a5v
    except Exception:
        import traceback as _s2_tb
        _s2_tb.print_exc()
        # Restore first: _A5_SAVED was captured while RUN_DINOV3 was still True,
        # so replaying it after the assignment would switch the stage back on.
        for _k, _v in globals().get('_A5_SAVED', {}).items():
            globals()[_k] = _v
        RUN_DINOV3 = False
        wall_log('stage 2 FAILED -- keeping the stage 1 submission; stage 3 still runs')

In [ ]:
if RUN_DINOV3:
  try:
    _a5_sub = pd.read_csv('/kaggle/working/submission.csv',
                          dtype={'StudyInstanceUID': str})
    assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
    A5_W = A5_W_SETTING
    if A5_W > 0:
        _a5_ours = np.stack([A5_PREDS[_u]
                             for _u in _a5_sub['StudyInstanceUID'].astype(str)])
        _a5_base_rank = _a5_sub[A5_LABELS].rank(method='average', pct=True)
        _a5_ours_rank = pd.DataFrame(_a5_ours, columns=A5_LABELS,
                                     index=_a5_sub.index).rank(method='average', pct=True)
        _a5_blend = (1.0 - A5_W) * _a5_base_rank + A5_W * _a5_ours_rank

        # A study whose slots were all empty comes back NaN. The source notebook
        # asserted here, which would abort the commit and lose the submission
        # entirely; instead those rows keep their stage 1 rank.
        _a5_bad = ~np.isfinite(_a5_blend.to_numpy())
        if _a5_bad.any():
            _a5_rows = int(_a5_bad.any(axis=1).sum())
            wall_log(f'stage 2: {_a5_rows} study/studies had no usable DINOv3 slots; '
                     f'they keep their stage 1 ranks')
            _a5_blend = _a5_blend.where(np.isfinite(_a5_blend), _a5_base_rank)

        assert np.isfinite(_a5_blend.to_numpy()).all(), 'stage 2 blend still non-finite'
        _a5_sub[A5_LABELS] = _a5_blend
        _a5_sub.to_csv('/kaggle/working/submission.csv', index=False)
        wall_log(f'stage 2 complete: DINOv3 blended at w={A5_W}')
  except Exception:
    import traceback as _s2_tb
    _s2_tb.print_exc()
    wall_log('stage 2 blend FAILED -- submission.csv still holds the stage 1 result')

## Stage 3 — RadImageNet

Frozen RadImageNet ResNet-50 plus five query heads, blended at `RAD_ALPHA` for ten of the twelve findings. Every weight file is SHA-256 verified against the published manifest before use.

In [ ]:
# Fixed five-fold RadImageNet inference; no fitting or weight search.
from __future__ import annotations
import contextlib as _rad_contextlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import shutil as _rad_shutil
import traceback as _rad_traceback
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath

import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50

_RAD_LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
_RAD_PLANES = ('Sagittal', 'Coronal', 'Axial')
_RAD_N_SLOT, _RAD_N_SLICE, _RAD_IMG = 3, 8, 224
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = 2048, 512
_RAD_SLICE_BAND = (0.12, 0.88)
_RAD_ALPHA = RAD_ALPHA
# RadImageNet is left out of these two findings entirely: it lowered them in our
# own gold audit, and the public 0.909 notebook (cf696666) excludes the same two.
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_FOLD_SHA256 = '1301603a060226c47c96be54d4c3618fee41f2e97f8f82d8f77a752819ffb7e3'
_RAD_CONFIG_HASH = '794a44d95a0ebb7096f17daa5a06dc191ec16c4d0835e69ac1868e82a7eb05dd'
_RAD_HEAD_SHA256 = {
    'rad_head_f0.pt': '0c92b27578e139cc35071a3f72ddd4e1a66106761225a7e011f076f37eb7051d',
    'rad_head_f1.pt': 'c29e60a99982d8dfb933a276f51c8b3a7d2e849649ed27a09affa822405fbd17',
    'rad_head_f2.pt': '40328ac7d72ca281e7e04438643100e99eb87fbe1e2b51965350b1a73a7b57ab',
    'rad_head_f3.pt': '6538a5faa92b61705727a14bd98c5ddce2989028dd8cccb261c47b2066fa5efa',
    'rad_head_f4.pt': '5bd126d68ddadd479a0eab6f0cdb17603060614a7eb14402e5b9945f651107af',
}
_RAD_FATSAT_OPTIONS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_RAD_FATSAT_PATTERN = _rad_re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)


def _rad_log(message):
    print(f'[Rad15] {message}', flush=True)


def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()


def _rad_find_competition():
    override = _rad_os.environ.get('RSNA_RAD_COMP_ROOT')
    candidates = [_RadPath(override)] if override else []
    candidates += [
        _RadPath('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        _RadPath('/kaggle/input/rsna-knee-abnormality-detection'),
    ]
    for candidate in candidates:
        if (candidate / 'test.csv').is_file() and (candidate / 'test_series').is_dir():
            return candidate
    base = _RadPath('/kaggle/input')
    if base.is_dir():
        for _r, _dirs, _files in _rad_os.walk(base):
            _dirs[:] = [_d for _d in _dirs
                        if _d not in ('train_series', 'test_series')]
            if 'test.csv' in _files and (_RadPath(_r) / 'test_series').is_dir():
                return _RadPath(_r)
    raise FileNotFoundError('Rad15 could not find competition test.csv/test_series')


def _rad_find_file(name, expected_sha=None, explicit_env=None):
    if explicit_env and _rad_os.environ.get(explicit_env):
        candidates = [_RadPath(_rad_os.environ[explicit_env])]
    else:
        candidates = []
        base = _RadPath('/kaggle/input')
        if base.is_dir():
            for root, dirs, files in _rad_os.walk(base):
                dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
                if name in files:
                    candidates.append(_RadPath(root) / name)
    if not candidates:
        raise FileNotFoundError(f'Rad15 missing input artifact {name}')
    for path in candidates:
        if expected_sha is None or _rad_sha256(path) == expected_sha:
            return path
    raise RuntimeError(f'Rad15 found {name}, but no copy has the required SHA-256')


def _rad_find_head_dir():
    override = _rad_os.environ.get('RSNA_RAD_HEAD_DIR')
    roots = [_RadPath(override)] if override else []
    if not roots:
        base = _RadPath('/kaggle/input')
        if base.is_dir():
            roots = []
            for _r, _dirs, _files in _rad_os.walk(base):
                _dirs[:] = [_d for _d in _dirs
                            if _d not in ('train_series', 'test_series')]
                if 'rad_heads_manifest.json' in _files:
                    roots.append(_RadPath(_r))
    for root in roots:
        manifest_path = root / 'rad_heads_manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = _rad_json.loads(manifest_path.read_text())
        if manifest.get('artifact_type') != 'rsna-radimagenet-foldsv1-heads':
            continue
        if manifest.get('fold_sha256') != _RAD_FOLD_SHA256:
            raise RuntimeError('Rad15 head manifest fold hash drift')
        if manifest.get('config_hash') != _RAD_CONFIG_HASH:
            raise RuntimeError('Rad15 head manifest config hash drift')
        if manifest.get('labels') != _RAD_LABELS:
            raise RuntimeError('Rad15 head manifest label order drift')
        return root, manifest
    raise FileNotFoundError('Rad15 five-head dataset is not attached')


def _rad_vector(value, length):
    try:
        result = _rad_np.asarray([float(item) for item in value], dtype=_rad_np.float64)
    except Exception:
        return None
    if len(result) < length or not _rad_np.isfinite(result[:length]).all():
        return None
    return result


def _rad_dicom_files(directory):
    return sorted(_RadPath(directory).glob('*.dcm'), key=lambda path: path.name)


def _rad_header(path):
    tags = [
        'ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
        'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'SeriesDescription',
        'SequenceName', 'ScanOptions',
    ]
    try:
        return _rad_pydicom.dcmread(
            str(path), stop_before_pixels=True, force=True, specific_tags=tags
        )
    except Exception:
        return None


def _rad_image_center_x(ds):
    if ds is None:
        return None
    ipp = _rad_vector(getattr(ds, 'ImagePositionPatient', None), 3)
    iop = _rad_vector(getattr(ds, 'ImageOrientationPatient', None), 6)
    spacing = _rad_vector(getattr(ds, 'PixelSpacing', None), 2)
    try:
        rows, cols = float(ds.Rows), float(ds.Columns)
    except Exception:
        return None
    if ipp is None or iop is None or spacing is None:
        return None
    center = ipp[:3] + iop[:3] * spacing[1] * cols / 2.0 + iop[3:6] * spacing[0] * rows / 2.0
    return float(center[0])


def _rad_is_fatsat(ds):
    if ds is None:
        return False
    description = (
        f"{getattr(ds, 'SeriesDescription', '') or ''} "
        f"{getattr(ds, 'SequenceName', '') or ''}"
    ).lower()
    description = _rad_re.sub(r'[_\-.]', ' ', description)
    options = getattr(ds, 'ScanOptions', None)
    if options is None:
        tokens = []
    elif isinstance(options, str):
        tokens = _rad_re.split(r'[|\\]', options)
    else:
        try:
            tokens = list(options)
        except TypeError:
            tokens = [options]
    option_match = any(str(token).strip().upper() in _RAD_FATSAT_OPTIONS for token in tokens)
    return bool(_RAD_FATSAT_PATTERN.search(description) or option_match)


def _rad_inspect_series(job):
    study, series, plane, directory = job
    files = _rad_dicom_files(directory)
    ds = _rad_header(files[len(files) // 2]) if files else None
    laterality = str(getattr(ds, 'Laterality', '') or '').strip().upper()[:1]
    laterality = laterality if laterality in ('L', 'R') else None
    return {
        'study': study, 'series': series, 'plane': plane,
        'directory': str(directory), 'n_files': len(files),
        'fatsat': _rad_is_fatsat(ds), 'laterality': laterality,
        'center_x': _rad_image_center_x(ds),
    }


def _rad_select_series(comp, studies):
    table = _rad_pd.read_csv(
        comp / 'test_series.csv',
        dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str},
    )
    table = table[table['StudyInstanceUID'].isin(studies)]
    root = comp / 'test_series'
    jobs = [
        (str(row.StudyInstanceUID), str(row.SeriesInstanceUID),
         str(row.Anatomical_Plane), root / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID))
        for row in table.itertuples(index=False)
    ]
    workers = max(1, min(16, _rad_os.cpu_count() or 1))
    with _RadThreadPool(max_workers=workers) as pool:
        inspected = list(pool.map(_rad_inspect_series, jobs))
    selected = {uid: [None] * _RAD_N_SLOT for uid in studies}
    tags = {uid: [] for uid in studies}
    centers = {uid: [] for uid in studies}
    for record in inspected:
        uid = record['study']
        if record['laterality']:
            tags[uid].append(record['laterality'])
        if record['center_x'] is not None:
            centers[uid].append(record['center_x'])
        if not record['fatsat'] or record['plane'] not in _RAD_PLANES:
            continue
        slot = _RAD_PLANES.index(record['plane'])
        current = selected[uid][slot]
        if current is None or record['n_files'] > current['n_files']:
            selected[uid][slot] = record
    sides = {}
    for uid in studies:
        if tags[uid]:
            sides[uid] = tags[uid][0]
        elif centers[uid]:
            middle = float(_rad_np.median(centers[uid]))
            sides[uid] = None if abs(middle) < 20.0 else ('R' if middle < 0 else 'L')
        else:
            sides[uid] = None
    return selected, sides


def _rad_order_files(directory):
    files = _rad_dicom_files(directory)
    rows = []
    for path in files:
        ds = _rad_header(path)
        key = None
        if ds is not None:
            ipp = _rad_vector(getattr(ds, 'ImagePositionPatient', None), 3)
            iop = _rad_vector(getattr(ds, 'ImageOrientationPatient', None), 6)
            if ipp is not None and iop is not None:
                key = float(_rad_np.dot(ipp[:3], _rad_np.cross(iop[:3], iop[3:6])))
            if key is None:
                try:
                    key = float(ds.InstanceNumber)
                except Exception:
                    pass
        rows.append((key, path))
    if any(key is None for key, _ in rows):
        return files
    return [path for _, path in sorted(rows, key=lambda item: item[0])]


def _rad_sample_indices(length):
    if length <= 0:
        return _rad_np.zeros(0, dtype=_rad_np.int64)
    lo = int(_RAD_SLICE_BAND[0] * (length - 1))
    hi = int(_RAD_SLICE_BAND[1] * (length - 1))
    if hi > lo:
        indices = _rad_np.unique(_rad_np.linspace(lo, hi, _RAD_N_SLICE).astype(_rad_np.int64))
    else:
        indices = _rad_np.array([length // 2], dtype=_rad_np.int64)
    while len(indices) < _RAD_N_SLICE:
        indices = _rad_np.append(indices, indices[-1])
    return indices[:_RAD_N_SLICE]


def _rad_read_series(record):
    files = _rad_order_files(record['directory'])
    indices = _rad_sample_indices(len(files))
    if not len(indices):
        return None
    images = []
    for index in indices:
        try:
            ds = _rad_pydicom.dcmread(str(files[int(index)]), force=True)
            image = ds.pixel_array.astype(_rad_np.float32)
            image = image * float(getattr(ds, 'RescaleSlope', 1) or 1)
            image = image + float(getattr(ds, 'RescaleIntercept', 0) or 0)
        except Exception:
            image = None
        images.append(image)
    valid = [index for index, image in enumerate(images) if image is not None]
    if not valid:
        return None
    for index, image in enumerate(images):
        if image is None:
            images[index] = images[min(valid, key=lambda other: abs(other - index))]
    shape = images[0].shape
    images = [image if image.shape == shape else _rad_np.zeros(shape, _rad_np.float32)
              for image in images]
    volume = _rad_np.stack(images).astype(_rad_np.float32, copy=False)
    lo, hi = _rad_np.percentile(volume, [1.0, 99.0])
    volume = _rad_np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0.0, 1.0)
    tensor = _rad_torch.from_numpy(_rad_np.ascontiguousarray(volume)).unsqueeze(0)
    resized = _rad_F.interpolate(
        tensor, size=(_RAD_IMG, _RAD_IMG), mode='bilinear', align_corners=False
    ).squeeze(0)
    return resized.mul(255).round().clamp(0, 255).to(_rad_torch.uint8).numpy()


def _rad_build_study(job):
    index, uid, records, side = job
    output = _rad_np.zeros((_RAD_N_SLOT, _RAD_N_SLICE, _RAD_IMG, _RAD_IMG), _rad_np.uint8)
    mask = _rad_np.zeros(_RAD_N_SLOT, _rad_np.uint8)
    for slot, record in enumerate(records):
        if record is None:
            continue
        image = _rad_read_series(record)
        if image is None:
            continue
        if side == 'R':
            image = image[::-1].copy() if record['plane'] == 'Sagittal' else image[:, :, ::-1].copy()
        output[slot] = image
        mask[slot] = 1
    return index, uid, output, mask


class _RadEncoder(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(*list(_rad_resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


class _RadHead(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_TOKEN_DIM),
            _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
        )
        self.plane = _rad_nn.Parameter(_rad_torch.randn(_RAD_N_SLOT, _RAD_HEAD_DIM) * 0.01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(_RAD_N_SLICE, _RAD_HEAD_DIM) * 0.01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.attn = _rad_nn.MultiheadAttention(_RAD_HEAD_DIM, 8, dropout=0.10, batch_first=True)
        self.fuse = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_HEAD_DIM * 4),
            _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM),
            _rad_nn.GELU(), _rad_nn.Dropout(0.15),
        )
        self.weight = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        if feature.shape[1:] != (_RAD_N_SLOT * _RAD_N_SLICE, _RAD_TOKEN_DIM):
            raise ValueError(f'Rad15 feature contract drift: {tuple(feature.shape)}')
        token = self.project(feature.float())
        token = token.view(len(token), _RAD_N_SLOT, _RAD_N_SLICE, _RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(
            query, token, token, key_padding_mask=key_padding, need_weights=False
        )[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat(
            [attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1
        ))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def _rad_load_models(device):
    encoder_path = _rad_find_file(
        'ResNet50.pt', _RAD_ENCODER_SHA256, explicit_env='RSNA_RAD_WEIGHT_PATH'
    )
    encoder = _RadEncoder()
    encoder.load_state_dict(
        _rad_torch.load(encoder_path, map_location='cpu', weights_only=True), strict=True
    )
    if sum(parameter.numel() for parameter in encoder.parameters()) != 23508032:
        raise RuntimeError('Rad15 encoder parameter-count drift')
    encoder.eval().to(device)
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    head_dir, manifest = _rad_find_head_dir()
    heads, observed_heads = [], {}
    for fold in range(5):
        name = f'rad_head_f{fold}.pt'
        path = head_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        observed = _rad_sha256(path)
        if observed != _RAD_HEAD_SHA256[name] or manifest['heads'][name]['sha256'] != observed:
            raise RuntimeError(f'Rad15 head hash drift for {name}')
        checkpoint = _rad_torch.load(path, map_location='cpu', weights_only=False)
        config = checkpoint.get('config', {})
        if checkpoint.get('fold') != fold or checkpoint.get('config_hash') != _RAD_CONFIG_HASH:
            raise RuntimeError(f'Rad15 checkpoint identity drift for fold {fold}')
        if config.get('gold_override') is not False or config.get('target_mode') != 'public3':
            raise RuntimeError(f'Rad15 checkpoint training contract drift for fold {fold}')
        if config.get('folds', {}).get('sha256') != _RAD_FOLD_SHA256:
            raise RuntimeError(f'Rad15 checkpoint fold hash drift for fold {fold}')
        head = _RadHead().to(device).eval()
        head.load_state_dict(checkpoint['state_dict'], strict=True)
        heads.append(head)
        observed_heads[name] = observed
    return encoder, heads, str(encoder_path), observed_heads


@_rad_torch.inference_mode()
def _rad_encode_block(encoder, pixels, slot_mask, device):
    n = len(pixels)
    features = _rad_np.zeros((n, _RAD_N_SLOT * _RAD_N_SLICE, _RAD_TOKEN_DIM), _rad_np.float16)
    token_mask = _rad_np.repeat(slot_mask[:, :, None], _RAD_N_SLICE, axis=2).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, _RAD_IMG, _RAD_IMG)
    batch = 96 if device.type == 'cuda' else 8
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = (_rad_torch.autocast('cuda', dtype=_rad_torch.float16)
               if device.type == 'cuda' else _rad_contextlib.nullcontext())
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('Rad15 non-finite encoder feature')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return features, token_mask.astype(_rad_np.float32)


@_rad_torch.inference_mode()
def _rad_predict_heads(heads, features, masks, device):
    feature = _rad_torch.from_numpy(features).to(device)
    mask = _rad_torch.from_numpy(masks).to(device)
    predictions = []
    # Heads were selected and rescored without autocast; preserve that contract.
    for head in heads:
        predictions.append(_rad_torch.sigmoid(head(feature, mask)).cpu().numpy())
    return predictions


def _rad_rank_columns(values):
    return _rad_pd.DataFrame(_rad_np.asarray(values, dtype=_rad_np.float64)).rank(
        method='average', pct=True
    ).to_numpy(_rad_np.float64)


def _rad_validate_submission(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('Rad15 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('Rad15 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError('Rad15 invalid submission values')


def _rad_main():
    started = _rad_time.time()
    comp = _rad_find_competition()
    work = _RadPath(_rad_os.environ.get('RSNA_RAD_OUTPUT_DIR', '/kaggle/working'))
    work.mkdir(parents=True, exist_ok=True)
    primary = work / 'submission.csv'
    preserved = work / 'submission_v8_before_rad.csv'
    audit_path = work / 'rad15_audit.json'
    if not primary.is_file():
        raise FileNotFoundError('Rad15 requires the completed V8 submission.csv')
    baseline = _rad_pd.read_csv(primary, dtype={'StudyInstanceUID': str})
    test = _rad_pd.read_csv(comp / 'test.csv', dtype={'StudyInstanceUID': str})
    studies = test['StudyInstanceUID'].astype(str).tolist()
    _rad_validate_submission(baseline, studies)
    _rad_shutil.copy2(primary, preserved)

    device = _rad_torch.device('cuda:0' if _rad_torch.cuda.is_available() else 'cpu')
    encoder, heads, encoder_path, head_hashes = _rad_load_models(device)
    selected, sides = _rad_select_series(comp, studies)
    available_slots = sum(record is not None for uid in studies for record in selected[uid])
    if available_slots < int(0.85 * len(studies) * _RAD_N_SLOT):
        raise RuntimeError(
            f'Rad15 only found {available_slots}/{len(studies) * _RAD_N_SLOT} fat-sat slots'
        )
    fold_predictions = [
        _rad_np.full((len(studies), len(_RAD_LABELS)), _rad_np.nan, _rad_np.float32)
        for _ in range(5)
    ]
    decode_workers = max(1, min(4, _rad_os.cpu_count() or 1))
    block_size = 32
    with _RadThreadPool(max_workers=decode_workers) as pool:
        for start in range(0, len(studies), block_size):
            block = studies[start:start + block_size]
            pixels = _rad_np.zeros(
                (len(block), _RAD_N_SLOT, _RAD_N_SLICE, _RAD_IMG, _RAD_IMG), _rad_np.uint8
            )
            slot_mask = _rad_np.zeros((len(block), _RAD_N_SLOT), _rad_np.uint8)
            jobs = [
                (index, uid, selected[uid], sides[uid])
                for index, uid in enumerate(block)
            ]
            for index, uid, image, mask in pool.map(_rad_build_study, jobs):
                if uid != block[index]:
                    raise RuntimeError('Rad15 decoder returned a misindexed study')
                pixels[index], slot_mask[index] = image, mask
            actual = pixels.reshape(len(block), _RAD_N_SLOT, -1).max(2) > 0
            if not _rad_np.array_equal(actual, slot_mask > 0):
                raise RuntimeError('Rad15 pixel/mask alignment failed')
            features, token_mask = _rad_encode_block(encoder, pixels, slot_mask, device)
            block_predictions = _rad_predict_heads(heads, features, token_mask, device)
            for fold, prediction in enumerate(block_predictions):
                fold_predictions[fold][start:start + len(block)] = prediction
            done = start + len(block)
            elapsed = _rad_time.time() - started
            eta = elapsed / done * (len(studies) - done) if done else 0
            _rad_log(f'{done:,}/{len(studies):,} studies; elapsed {elapsed/60:.1f}m, eta {eta/60:.1f}m')
            del pixels, slot_mask, features, token_mask, block_predictions
            _rad_gc.collect()
    if not all(_rad_np.isfinite(prediction).all() for prediction in fold_predictions):
        raise RuntimeError('Rad15 incomplete/non-finite fold predictions')

    # Required estimator: rank every fold first, then average the five fold ranks.
    rad_rank = _rad_np.mean(
        _rad_np.stack([_rad_rank_columns(prediction) for prediction in fold_predictions]),
        axis=0,
    )
    v8_rank = _rad_rank_columns(baseline[_RAD_LABELS].to_numpy(_rad_np.float64))
    candidate = baseline.copy()
    _rad_alpha_vec = _rad_np.array(
        [0.0 if _t in _RAD_EXCLUDE else _RAD_ALPHA for _t in _RAD_LABELS],
        dtype=_rad_np.float64)[None, :]
    candidate[_RAD_LABELS] = (1.0 - _rad_alpha_vec) * v8_rank + _rad_alpha_vec * rad_rank
    _rad_validate_submission(candidate, studies)
    temporary = primary.with_suffix('.csv.tmp')
    candidate.to_csv(temporary, index=False)
    _rad_os.replace(temporary, primary)
    audit = {
        'status': 'FIXED_RAD15_WRITTEN',
        'formula': f'{1-_RAD_ALPHA:g}*rank(V8)+{_RAD_ALPHA:g}*mean_fold(rank(Rad_fold)); alpha=0 for {list(_RAD_EXCLUDE)}',
        'alpha': _RAD_ALPHA,
        'excluded_targets': list(_RAD_EXCLUDE),
        'weight_policy': 'one fixed weight shared by every blended finding; two findings excluded outright; no test-time selection',
        'studies': len(studies),
        'available_slots': int(available_slots),
        'encoder_path': encoder_path,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'head_sha256': head_hashes,
        'fold_sha256': _RAD_FOLD_SHA256,
        'config_hash': _RAD_CONFIG_HASH,
        'baseline_sha256': _rad_sha256(preserved),
        'submission_sha256': _rad_sha256(primary),
        'elapsed_seconds': _rad_time.time() - started,
    }
    temporary_audit = audit_path.with_suffix('.json.tmp')
    temporary_audit.write_text(_rad_json.dumps(audit, indent=2, sort_keys=True) + '\n')
    _rad_os.replace(temporary_audit, audit_path)
    _rad_log(f"wrote fixed Rad15 submission ({audit['elapsed_seconds']/60:.1f}m)")



# The source notebook re-raised on any Rad15 failure. On Kaggle an uncaught
# exception fails the commit, which means no submission at all -- strictly worse
# than shipping the stage 2 result. Restore, log loudly, and carry on.
if not GPU_OK:
    wall_log('SKIPPING stage 3 (RadImageNet): no usable GPU')
elif (time.time() - WALL_T0) >= RAD_START_CUTOFF_S or wall_left() < 600:
    wall_log('SKIPPING stage 3 (RadImageNet): wall-clock budget spent, stage 2 output stands')
else:
    try:
        _rad_main()
        wall_log(f'stage 3 complete: RadImageNet blended at alpha={_RAD_ALPHA}')
    except Exception:
        _rad_traceback.print_exc()
        _rad_work = _RadPath(_rad_os.environ.get('RSNA_RAD_OUTPUT_DIR', '/kaggle/working'))
        _rad_saved = _rad_work / 'submission_v8_before_rad.csv'
        if _rad_saved.is_file():
            _rad_shutil.copy2(_rad_saved, _rad_work / 'submission.csv')
            wall_log('stage 3 FAILED -- restored the stage 2 submission (see traceback above)')
        else:
            wall_log('stage 3 FAILED before it touched submission.csv (see traceback above)')

## Final check

Validates the file against the competition contract before the commit ends.

In [ ]:
import pandas as _v_pd, numpy as _v_np
from pathlib import Path as _v_Path

_v_targets = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
              'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
              'Contusion', 'Fracture']

_v_root = None
for _c in (_v_Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
           _v_Path('/kaggle/input/rsna-knee-abnormality-detection')):
    if (_c / 'test.csv').is_file():
        _v_root = _c
        break
_v_test = _v_pd.read_csv(_v_root / 'test.csv', dtype={'StudyInstanceUID': str})
_v_sub = _v_pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})

assert list(_v_sub.columns) == ['StudyInstanceUID'] + _v_targets, 'column contract broken'
assert len(_v_sub) == len(_v_test), f'{len(_v_sub)} rows vs {len(_v_test)} in test.csv'
assert _v_sub['StudyInstanceUID'].is_unique, 'duplicate StudyInstanceUID'
assert set(_v_sub['StudyInstanceUID']) == set(_v_test['StudyInstanceUID']), 'UID set mismatch'
_v_vals = _v_sub[_v_targets].to_numpy(float)
assert _v_np.isfinite(_v_vals).all(), 'non-finite prediction'
assert _v_vals.min() >= 0.0 and _v_vals.max() <= 1.0, 'prediction outside [0, 1]'
assert _v_vals.std() > 0, 'every prediction is identical -- this is the 0.5 benchmark'

wall_log(f'submission.csv VALID: {_v_sub.shape[0]} rows x {len(_v_targets)} targets')
print(f'total wall clock: {(time.time() - WALL_T0) / 60:.1f} min')
_v_sub.head()